# Citi Bike: Research Notebook — Fleet-Conserving Targets
Step 6 adds inventory allocation based on historical event order and the existing fleet size. Step 8 uses Gurobi only. Prepared inputs are bundled, so you can run Step 8 without repeating prediction. Steps 1–5 document the historical analysis; their superseded targets are references only.


## Step 1: Research scope

| Item | Definition |
|---|---|
| Study area | Citi Bike stations on Manhattan's main island; exclude other boroughs and separate islands |
| Regions | Lower: south of 14th Street; Midtown: 14th–59th Street; Upper: north of 59th Street |
| Boundaries | Street centerlines and the island polygon; a shared boundary belongs to one region only |
| Station scope | All historical station IDs in the study area, without a sample-size cap; audit renumbering, openings and closures |
| Historical data | All CSV parts of the official July, August and September 2026 trip ZIPs |
| Analysis dates | July 2–September 29, 2026; trim the endpoint days to reduce cross-month omissions |
| Weekdays | Monday–Friday, including public holidays |
| Morning peak | 06:00 <= local New York time < 10:00 |
| Pickups P | Completed trip starts, counted by starting station and time |
| Returns R | Completed trip ends, counted by ending station and time; include arrivals from outside the area |
| Demand units | Events per station per weekday morning peak |
| Static demand | One fixed P and R estimate per station over the selected forecast dates |

The 14th/59th Street division is an operational research definition, not the only official neighborhood classification. Later steps fix the inventory case at midnight on October 28 and add optimization.


### Data sources and roles

- Official trip records support the regional comparison and pickup/return prediction. Timely inventory charts on ctbk.dev do not imply that monthly trip records have no publication delay.
- Historical GBFS records from ctbk.dev provide the midnight inventory case, timestamps and status checks.
- Station metadata and historical endpoint coordinates support identity, location and capacity checks. A current roster alone cannot identify every historical station.
- Main-island and street boundaries define the geographic scope.

Sources: [Citi Bike system data](https://citibikenyc.com/system-data), [ctbk.dev](https://ctbk.dev/), [ctbk data pipeline](https://github.com/hudcostreets/ctbk.dev/blob/main/docs/pipeline.md).


### Selecting a region from data
Compare historical observations before fitting prediction models:

1. Station counts, dates and coverage.
2. Total pickup/return activity and activity per station.
3. Station-level net flow P−R, its direction and spatial distribution.
4. Both regional net flow and the sum of station-level absolute net flows, so opposing flows do not cancel invisibly.

These statistics describe demand patterns and research relevance. They do not prove optimal rebalancing benefits; the largest trip total alone is insufficient.


## Step 2: Read data, audit quality and map the three regions
Check whether all files were read, which station IDs appear, and whether the regions cover the main island without overlap. Historical IDs are statistical units, not automatically distinct physical stations. Audit large location changes and cross-boundary coordinates separately.

Boundary sources: NYC DCP [Borough Boundaries](https://data.cityofnewyork.us/City-Government/Borough-Boundaries/gthc-hcne) and NYC [Centerline](https://data.cityofnewyork.us/resource/inkn-q76z.geojson). Use the largest connected Manhattan land polygon; exclude Roosevelt Island, Governors Island and Marble Hill.

Interpolate the E/W 14 ST, E/W 59 ST and CENTRAL PARK S centerlines from west to east and extend their ends to the shoreline. Where branches share an x-coordinate, use the median y-coordinate. This reproducible approximation is not a claim of exact street-side classification. List stations within 30 m of either dividing line for review; shared boundaries belong to the northern region.


In [ ]:
# Uncomment the next line if dependencies are missing.
# %pip install pandas numpy matplotlib requests shapely pyproj
from pathlib import Path
import json, zipfile, urllib.request, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from shapely.geometry import shape, Point, LineString
from shapely.ops import transform
from pyproj import Transformer
from IPython.display import display, Markdown
ROOT=Path.cwd(); RAW=ROOT/'data/raw'; GEO=ROOT/'data/geography'; OUT=ROOT/'outputs_stage2'
for folder in [RAW,GEO,OUT]: folder.mkdir(parents=True,exist_ok=True)
MONTHS=['202607','202608','202609']
START,END=pd.Timestamp('2026-07-02'),pd.Timestamp('2026-09-29')
REGION_ORDER=['Lower Manhattan','Midtown','Upper Manhattan']


In [ ]:
# Download geography once; keep raw files in data/geography.
geo_sources={
 'manhattan_borough.geojson':('https://data.cityofnewyork.us/resource/gthc-hcne.geojson',{'$where':"boroname='Manhattan'"}),
 'street_14.geojson':('https://data.cityofnewyork.us/resource/inkn-q76z.geojson',{'$where':"boroughcode='1' AND (full_street_name='W  14 ST' OR full_street_name='E  14 ST')",'$limit':5000}),
 'street_59.geojson':('https://data.cityofnewyork.us/resource/inkn-q76z.geojson',{'$where':"boroughcode='1' AND (full_street_name='W  59 ST' OR full_street_name='E  59 ST' OR full_street_name='CENTRAL PARK S')",'$limit':5000})}
for filename,(url,params) in geo_sources.items():
    path=GEO/filename
    if not path.exists():
        response=requests.get(url,params=params,timeout=60);response.raise_for_status()
        path.write_text(response.text,encoding='utf-8')
    if not json.loads(path.read_text())['features']:raise ValueError('No geographic features returned: '+filename)
project=Transformer.from_crs(4326,32618,always_xy=True).transform
borough=shape(json.loads((GEO/'manhattan_borough.geojson').read_text())['features'][0]['geometry'])
land=transform(project,borough)
island=max(land.geoms,key=lambda g:g.area) if land.geom_type=='MultiPolygon' else land
street_lines={}; vertices={}
for number in [14,59]:
    lines=[]
    for f in json.loads((GEO/f'street_{number}.geojson').read_text())['features']:
        g=transform(project,shape(f['geometry']))
        lines.extend(list(g.geoms) if g.geom_type=='MultiLineString' else [g])
    points=np.vstack([np.asarray(g.coords) for g in lines])
    v=pd.DataFrame(points,columns=['x','y']).groupby('x',as_index=False).y.median().sort_values('x')
    vertices[number]=v
    street_lines[number]=lines

def boundary_y(x,number):
    v=vertices[number];xx=v.x.to_numpy();yy=v.y.to_numpy();x=np.asarray(x)
    out=np.interp(x,xx,yy)
    # Extend endpoints using an approximately 300 m direction to reduce short-segment noise.
    left=min(np.searchsorted(xx,xx[0]+300),len(xx)-1)
    right=max(np.searchsorted(xx,xx[-1]-300),0)
    out=np.where(x<xx[0],yy[0]+(x-xx[0])*(yy[left]-yy[0])/(xx[left]-xx[0]),out)
    out=np.where(x>xx[-1],yy[-1]+(x-xx[-1])*(yy[-1]-yy[right])/(xx[-1]-xx[right]),out)
    return out
xs=np.linspace(island.bounds[0]-100,island.bounds[2]+100,1000)
assert (boundary_y(xs,59)>boundary_y(xs,14)).all(), 'Boundary lines intersect; review the geometry.'
boundaries={k:LineString(np.column_stack([xs,boundary_y(xs,k)])) for k in [14,59]}
print('Main-island area (square km):',round(island.area/1e6,2))


### 2.1 Read every month and CSV part
Deduplicate by ride_id within and across months. Missing IDs cannot be reliably deduplicated and are excluded separately. Validate the start and end independently, retaining a usable endpoint even when the other is missing.

Include cross-region trips. Represent station locations by event-weighted mean coordinates over valid endpoints. Review IDs with coordinate spans above 100 m or coordinates crossing regions; these checks cannot automatically distinguish relocation from renumbering.

Daily system activity does not establish daily operation at every station. Retain first/last observations and active-day counts; do not silently treat unobserved station-days as zero-demand operating days.


In [ ]:
# Bind cache validity to raw file sizes and analysis dates; avoid rereading every ZIP.
cache_files=['source_quality.csv','all_station_geography_audit.csv','station_daily_activity.csv','peak_counts_all_stations.csv']
signature={'version':1,'start':str(START.date()),'end':str(END.date()),'raw_sizes':{m:(RAW/f'{m}-citibike-tripdata.zip').stat().st_size if (RAW/f'{m}-citibike-tripdata.zip').exists() else None for m in MONTHS}}
manifest=OUT/'cache_manifest.json'
cache_ok=manifest.exists() and all((OUT/f).exists() for f in cache_files)
if cache_ok:
    old=json.loads(manifest.read_text())
    # Bundled caches support missing local raw ZIPs; existing raw ZIP sizes must match.
    cache_ok=old['version']==signature['version'] and old['start']==signature['start'] and old['end']==signature['end'] and all(size is None or size==old['raw_sizes'][m] for m,size in signature['raw_sizes'].items())
if cache_ok:
    quality=pd.read_csv(OUT/'source_quality.csv')
    stations=pd.read_csv(OUT/'all_station_geography_audit.csv',dtype={'station_id':str},parse_dates=['first_date','last_date'])
    all_days=pd.read_csv(OUT/'station_daily_activity.csv',dtype={'station_id':str},parse_dates=['date'])
    peak_counts=pd.read_csv(OUT/'peak_counts_all_stations.csv',dtype={'station_id':str},parse_dates=['date'])
    print('Loaded cached real-data processing results; raw ZIPs were not reread.')
    display(quality)
else:
    station_parts=[]; endpoint_parts=[]; audit=[]; seen_previous=set(); all_day_parts=[]
    for month in MONTHS:
        path=RAW/f'{month}-citibike-tripdata.zip'
        url=f'https://tripdata.s3.amazonaws.com/{path.name}'
        if not path.exists():
            print('Downloading',month,flush=True);tmp=path.with_suffix('.part')
            urllib.request.urlretrieve(url,tmp)
            if not zipfile.is_zipfile(tmp):raise ValueError('Downloaded file is not a ZIP.')
            tmp.replace(path)
        total=missing_id=duplicates=bad_time=missing_sid=bad_coords=0;seen_month=set()
        with zipfile.ZipFile(path) as z:
            members=[s for s in z.namelist() if s.endswith('.csv') and not s.startswith('__MACOSX/')]
            if not members:raise ValueError('ZIP contains no CSV files.')
            for member in members:
                with z.open(member) as stream:
                    columns=['ride_id','started_at','ended_at','start_station_id','end_station_id','start_station_name','end_station_name','start_lat','start_lng','end_lat','end_lng']
                    for chunk in pd.read_csv(stream,usecols=columns,dtype=str,chunksize=200000):
                        total+=len(chunk);missing_id+=int(chunk.ride_id.isna().sum());chunk=chunk.dropna(subset=['ride_id'])
                        dup=chunk.ride_id.map(seen_month.__contains__)|chunk.ride_id.map(seen_previous.__contains__)|chunk.ride_id.duplicated()
                        duplicates+=int(dup.sum());seen_month.update(chunk.ride_id);chunk=chunk.loc[~dup]
                        for prefix,tc,flow in [('start','started_at','P'),('end','ended_at','R')]:
                            t=pd.to_datetime(chunk[tc],errors='coerce',format='mixed')
                            sid=chunk[prefix+'_station_id'].str.strip().replace('',np.nan)
                            lat=pd.to_numeric(chunk[prefix+'_lat'],errors='coerce');lon=pd.to_numeric(chunk[prefix+'_lng'],errors='coerce')
                            bad_time+=int(t.isna().sum());missing_sid+=int(sid.isna().sum())
                            valid_coord=lat.between(40,42)&lon.between(-75,-72);bad_coords+=int((~valid_coord).sum())
                            period=t.dt.normalize().between(START,END)
                            q=pd.DataFrame({'date':t.dt.normalize(),'station_id':sid,'name':chunk[prefix+'_station_name'],'lat':lat,'lon':lon})
                            valid=period & sid.notna() & valid_coord & t.notna();q=q.loc[valid].copy()
                            # Discover stations using all-day activity, including stations with no morning trips.
                            summary=q.groupby('station_id').agg(n=('lat','size'),lat_sum=('lat','sum'),lon_sum=('lon','sum'),lat_min=('lat','min'),lat_max=('lat','max'),lon_min=('lon','min'),lon_max=('lon','max'),first_date=('date','min'),last_date=('date','max'),name=('name','first')).reset_index()
                            station_parts.append(summary)
                            allday=q.groupby(['date','station_id']).size().rename('events').reset_index();all_day_parts.append(allday)
                            peak=q[t.loc[q.index].dt.hour.between(6,9)].groupby(['date','station_id']).size().rename('count').reset_index();peak['flow']=flow;endpoint_parts.append(peak)
        seen_previous.update(seen_month)
        audit.append({'month':month,'csv_parts':len(members),'raw_rows':total,'missing_ride_id':missing_id,'duplicate_rides':duplicates,'bad_endpoint_time':bad_time,'missing_endpoint_station_id':missing_sid,'invalid_endpoint_coordinates':bad_coords,'zip_bytes':path.stat().st_size})
        print(month,'Read all CSV parts:',total,'rows',flush=True)
    # Missingness counts are overlapping endpoint counts, not additive excluded-trip totals.
    quality=pd.DataFrame(audit);display(quality)
    s=pd.concat(station_parts)
    stations=s.groupby('station_id').agg(n=('n','sum'),lat_sum=('lat_sum','sum'),lon_sum=('lon_sum','sum'),lat_min=('lat_min','min'),lat_max=('lat_max','max'),lon_min=('lon_min','min'),lon_max=('lon_max','max'),first_date=('first_date','min'),last_date=('last_date','max'),name=('name','first')).reset_index()
    stations['lat']=stations.lat_sum/stations.n;stations['lon']=stations.lon_sum/stations.n
    stations['coordinate_span_m']=np.hypot((stations.lat_max-stations.lat_min)*111320,(stations.lon_max-stations.lon_min)*111320*np.cos(np.deg2rad(stations.lat)))
    all_days=pd.concat(all_day_parts).groupby(['date','station_id']).events.sum().reset_index()
    active=all_days.groupby('station_id').date.nunique();stations['active_days']=stations.station_id.map(active)
    peak_counts=pd.concat(endpoint_parts).groupby(['date','station_id','flow'])['count'].sum().reset_index()
    quality.to_csv(OUT/'source_quality.csv',index=False)
    peak_counts.to_csv(OUT/'peak_counts_all_stations.csv',index=False)
    all_days.to_csv(OUT/'station_daily_activity.csv',index=False)
    # Release raw IDs and temporary aggregates; subsequent stages use smaller tables.
    del seen_previous,seen_month,station_parts,endpoint_parts,all_day_parts,s

### 2.2 Main-island membership and boundary review
Keep every historical ID classified inside the main island. Do not snap nearby offshore points into the polygon; list them separately. Exclusions do not establish complete coverage.


In [ ]:
x,y=project(stations.lon.to_numpy(),stations.lat.to_numpy());stations['x']=x;stations['y']=y
points=[Point(a,b) for a,b in zip(x,y)]
stations['on_main_island']=[island.covers(p) for p in points]
stations['shore_distance_m']=[island.boundary.distance(p) for p in points]
stations['region']=np.where(y<boundary_y(x,14),'Lower Manhattan',np.where(y<boundary_y(x,59),'Midtown','Upper Manhattan'))
stations.loc[~stations.on_main_island,'region']='Outside main island'
stations['boundary_distance_m']=[min(boundaries[14].distance(p),boundaries[59].distance(p)) for p in points]
stations['needs_coordinate_review']=stations.coordinate_span_m.gt(100)
stations['near_region_boundary']=stations.boundary_distance_m.le(30)&stations.on_main_island
main_stations=stations[stations.on_main_island].copy()
assert main_stations.station_id.is_unique
assert main_stations.region.isin(REGION_ORDER).all()
# Flag coordinate bounds crossing region or shore boundaries for raw-coordinate review.
cross=[]
for s in main_stations.itertuples():
    cx,cy=project(np.array([s.lon_min,s.lon_max,s.lon_min,s.lon_max]),np.array([s.lat_min,s.lat_min,s.lat_max,s.lat_max]))
    labels=np.where(cy<boundary_y(cx,14),'Lower Manhattan',np.where(cy<boundary_y(cx,59),'Midtown','Upper Manhattan'))
    cross.append(len(set(labels))>1 or any(not island.covers(Point(a,b)) for a,b in zip(cx,cy)))
main_stations['coordinate_extent_crosses_boundary']=cross
review=main_stations[main_stations.needs_coordinate_review|main_stations.near_region_boundary|main_stations.coordinate_extent_crosses_boundary]
shore_review=stations[(~stations.on_main_island)&stations.shore_distance_m.le(50)]
region_audit=main_stations.groupby('region').agg(station_ids=('station_id','size'),median_active_days=('active_days','median'),coordinate_span_over_100m=('needs_coordinate_review','sum'),near_split_30m=('near_region_boundary','sum')).reindex(REGION_ORDER)
display(region_audit);print('Boundary/location IDs requiring review:',len(review),'Offshore IDs within 50 m of shore:',len(shore_review))
coverage=all_days[all_days.station_id.isin(main_stations.station_id)].groupby('date').events.sum().reindex(pd.date_range(START,END),fill_value=0)
print('Dates:',len(coverage),'Weekdays:',int((coverage.index.dayofweek<5).sum()),'Main-island dates without valid events:',int(coverage.eq(0).sum()))
if coverage.eq(0).any():raise ValueError('A date has no main-island events; stop and investigate.')
stations.to_csv(OUT/'all_station_geography_audit.csv',index=False)
main_stations.to_csv(OUT/'manhattan_station_regions.csv',index=False)
region_audit.to_csv(OUT/'region_coverage_summary.csv')
review.to_csv(OUT/'boundary_coordinate_review.csv',index=False)
shore_review.to_csv(OUT/'shoreline_outside_review.csv',index=False)
coverage.rename('valid_endpoint_events').to_csv(OUT/'daily_coverage.csv')
display(review[['station_id','name','region','coordinate_span_m','boundary_distance_m']].round(1).head(20))

# Flag numeric-format aliases; do not merge without checking physical identity.
from decimal import Decimal, InvalidOperation
import re
def decimal_key(value):
    if not re.fullmatch(r'[0-9]+(?:\.[0-9]+)?',value):return value
    try:return format(Decimal(value).normalize(),'f')
    except InvalidOperation:return value
main_stations['numeric_id_key']=main_stations.station_id.map(decimal_key)
alias_review=main_stations[main_stations.numeric_id_key.duplicated(keep=False)].sort_values('numeric_id_key')
alias_review[['station_id','numeric_id_key','name','lat','lon','region']].to_csv(OUT/'station_id_alias_review.csv',index=False)
print('IDs with numeric-format differences:',len(alias_review))
display(alias_review[['station_id','numeric_id_key','name','region']])
# Create the cache signature only after processing completes.
if not cache_ok:manifest.write_text(json.dumps(signature,indent=2),encoding='utf-8')

# Merge verified numeric-format aliases with matching names/coordinates; preserve suffixed IDs.
main_stations['canonical_station_id']=main_stations.station_id
alias_decisions=[]
for key,g in alias_review.groupby('numeric_id_key'):
    same_name=g.name.fillna('').str.strip().nunique()==1
    max_distance=np.hypot(g.x.max()-g.x.min(),g.y.max()-g.y.min())
    ok=same_name and max_distance<=5
    canonical=sorted(g.station_id,key=lambda v:(-len(v),v))[0]
    if ok:main_stations.loc[g.index,'canonical_station_id']=canonical
    for sid in g.station_id:alias_decisions.append({'station_id':sid,'canonical_station_id':canonical if ok else sid,'merged_format_alias':ok,'coordinate_distance_m':max_distance})
pd.DataFrame(alias_decisions).to_csv(OUT/'alias_resolution.csv',index=False)
canonical_map=main_stations.set_index('station_id').canonical_station_id
canonical_stations=main_stations.drop_duplicates('canonical_station_id').copy()
canonical_stations['station_id']=canonical_stations.canonical_station_id
main_stations.to_csv(OUT/'manhattan_station_regions.csv',index=False)
canonical_stations.to_csv(OUT/'manhattan_canonical_stations.csv',index=False)
for frame,value,filename in [(peak_counts,'count','manhattan_daily_peak_counts.csv'),(all_days,'events','manhattan_daily_activity.csv')]:
    d=frame[frame.station_id.isin(canonical_map.index)].copy()
    d['station_id']=d.station_id.map(canonical_map)
    keys=['date','station_id']+(['flow'] if 'flow' in d else [])
    d=d.groupby(keys,as_index=False)[value].sum()
    d.to_csv(OUT/filename,index=False)
region_audit['after_verified_format_merge']=canonical_stations.groupby('region').size()
region_audit.to_csv(OUT/'region_coverage_summary.csv')
display(region_audit)
print('Only verified format aliases are merged; renumbering and relocation are not fully resolved.')


In [ ]:
colors={'Lower Manhattan':'#2b8cbe','Midtown':'#e69f00','Upper Manhattan':'#009e73'}
fig,ax=plt.subplots(figsize=(9,12))
bx,by=island.exterior.xy;ax.fill(np.array(bx)/1000,np.array(by)/1000,color='#f1f3f5',edgecolor='#59636b',lw=0.7)
for region in REGION_ORDER:
    d=canonical_stations[canonical_stations.region==region]
    ax.scatter(d.x/1000,d.y/1000,s=10,color=colors[region],label=f'{region}: {len(d)} station IDs',alpha=.8)
for number in [14,59]:
    clipped=boundaries[number].intersection(island)
    for segment in (list(clipped.geoms) if clipped.geom_type=='MultiLineString' else [clipped]):
        if segment.geom_type=='LineString':
            xx,yy=segment.xy;ax.plot(np.array(xx)/1000,np.array(yy)/1000,'k--',lw=1.2)
    v=vertices[number];ax.text(v.x.mean()/1000,v.y.mean()/1000,f' {number}th Street',fontsize=10,bbox=dict(facecolor='white',alpha=.8,edgecolor='none'))
ax.set_aspect('equal');ax.set_xlabel('UTM easting (km)');ax.set_ylabel('UTM northing (km)');ax.set_title('Manhattan main island: station IDs after verified format merge\nJuly–September 2026')
ax.legend(loc='upper left');plt.tight_layout();fig.savefig(OUT/'manhattan_regions.png',dpi=160);plt.show()
display(Markdown('Colors indicate research regions. Dashed lines interpolate street centerlines and extend to the shore. This map checks coverage and assignment, not relative demand; consult the review tables for boundary and moving stations.'))
fig,ax=plt.subplots(figsize=(10,3));coverage.plot(ax=ax,color='#365c8d');ax.set_ylabel('Valid endpoint events / day');ax.set_title('Main-island data coverage — all hours');plt.tight_layout();plt.show()
display(Markdown('The curve detects gaps in date coverage. Starts and ends are separate events, so this is not the ride count. Low values can reflect genuine low activity; records do not prove continuous station operation.'))
print('Step 2 complete: data and geography checked; region selection and model training follow.')


### Data audit results
All 16 CSV files from July–September 2026 were read, covering 15,222,228 rides. July 2–September 29 contains 64 weekdays. No date has zero main-island events, but this does not prove daily operation at each station. Merge numeric-format aliases only after checking names and coordinates; preserve suffixed IDs.


## Step 3: Compare the three regions
Use only the 43 weekdays from July 2–August 31 for regional selection. Reserve September for validation and testing. Geographic identity tables support retrospective research preparation.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
BASE3=Path.cwd(); INPUT3=BASE3/'outputs_stage2'; OUT3=BASE3/'outputs_stage3';OUT3.mkdir(exist_ok=True)
ORDER3=['Lower Manhattan','Midtown','Upper Manhattan']
COLORS3={'Lower Manhattan':'#2b8cbe','Midtown':'#e69f00','Upper Manhattan':'#009e73'}
SELECT_START3=pd.Timestamp('2026-07-02'); SELECT_END3=pd.Timestamp('2026-08-31')
DAYS3=pd.bdate_range(SELECT_START3,SELECT_END3)
station3=pd.read_csv(INPUT3/'manhattan_canonical_stations.csv',dtype={'station_id':str,'canonical_station_id':str})
identity3=pd.read_csv(INPUT3/'manhattan_station_regions.csv',dtype={'station_id':str,'canonical_station_id':str})
peak3=pd.read_csv(INPUT3/'manhattan_daily_peak_counts.csv',dtype={'station_id':str},parse_dates=['date'])
activity3=pd.read_csv(INPUT3/'manhattan_daily_activity.csv',dtype={'station_id':str},parse_dates=['date'])
review_raw3=pd.read_csv(INPUT3/'boundary_coordinate_review.csv',dtype={'station_id':str})
assert len(DAYS3)==43 and station3.station_id.is_unique
period3=activity3[activity3.date.isin(DAYS3)]
observed_days3=period3.groupby('station_id').date.nunique()
selection_stations3=station3[station3.station_id.isin(observed_days3.index)].copy()
selection_stations3['days_with_any_event']=selection_stations3.station_id.map(observed_days3)
new_later3=station3[~station3.station_id.isin(observed_days3.index)]
print('Selection weekdays:',len(DAYS3),'Analysis IDs observed in the selection period:',len(selection_stations3),'IDs first appearing later:',len(new_later3))
display(new_later3[['station_id','name','region']])
new_later3.to_csv(OUT3/'outside_selection_period_stations.csv',index=False)


### 3.1 Boundary and coordinate ambiguities
Thirteen raw records map to 12 analysis IDs after verified format merges. Eleven are near 14th or 59th Street. Keep their coordinate-based assignments rather than guessing from station names. Recompute after assigning all near-boundary stations north, assigning them south, and excluding them.

ID `5914.08` spans approximately 362 m. Its coordinate bounds do not cross a region or island boundary, so the main comparison retains its Lower Manhattan assignment. This does not verify the exact location or explain the movement. Also test excluding this ID; date-specific coordinates require review before routing.


In [ ]:
# Map raw review records to verified analysis IDs; preserve .10 as text.
idmap3=identity3.set_index('station_id').canonical_station_id
review_raw3['canonical_id']=review_raw3.station_id.map(idmap3)
assert review_raw3.canonical_id.notna().all()
review3=review_raw3.groupby('canonical_id').agg(boundary_distance_m=('boundary_distance_m','min'),coordinate_span_m=('coordinate_span_m','max'),extent_crosses_boundary=('coordinate_extent_crosses_boundary','max')).reset_index().rename(columns={'canonical_id':'station_id'})
review3=review3.merge(station3[['station_id','name','region','lat','lon']],on='station_id',validate='one_to_one')
review3['near_split']=review3.boundary_distance_m<=30
review3['position_uncertain']=review3.coordinate_span_m>100
# Identify the nearest dividing street without changing Step 2 assignments.
from shapely.geometry import shape,Point
from shapely.ops import transform,unary_union
from pyproj import Transformer
project3=Transformer.from_crs(4326,32618,always_xy=True).transform
street_geometries3={}
for k in [14,59]:
    raw=json.loads((BASE3/f'data/geography/street_{k}.geojson').read_text())
    street_geometries3[k]=unary_union([transform(project3,shape(f['geometry'])) for f in raw['features']])
nearest3=[]
for row in review3.itertuples():
    point=Point(*project3(row.lon,row.lat))
    nearest3.append(min([14,59],key=lambda k:point.distance(street_geometries3[k])))
review3['nearest_street']=nearest3
review3['decision']=np.where(review3.near_split,'Retain coordinate assignment; test north/south reassignment and exclusion separately.','Retain region; verify location later and test exclusion separately.')
review3.to_csv(OUT3/'review_decisions.csv',index=False,encoding='utf-8-sig')
display(review3[['station_id','name','region','boundary_distance_m','coordinate_span_m','decision']].round(1))
print('Review IDs:',len(review3),'Near a boundary:',int(review3.near_split.sum()),'Unverified location:',int(review3.position_uncertain.sum()))


### 3.2 Comparison metrics
Compute each station's mean pickups P and returns R over the 43 selection weekdays.

| Metric | Calculation | Interpretation and limitation |
|---|---|---|
| Total activity A | Σ(P+R) | Endpoint events per morning; an internal trip may contribute two endpoints |
| Activity per station | A / analysis ID count | Separates scale from typical station activity; not demand per unit area |
| Absolute net flow B | Σ\|P−R\| | Station directionality, not required truck transfers |
| Absolute net flow per station | B / analysis ID count | Average directional imbalance |
| Regional net flow | Σ(P−R) | Positive means net departures; opposing station flows may cancel |
| Opposing-flow scale | min(Σmax(P−R,0), Σmax(R−P,0)) | Whether both flow directions have substantial scale; not proven transferable inventory |


In [ ]:
selection_peak3=peak3[peak3.date.isin(DAYS3)&peak3.station_id.isin(selection_stations3.station_id)].copy()
index3=pd.MultiIndex.from_product([DAYS3,selection_stations3.station_id,['P','R']],names=['date','station_id','flow'])
panel3=selection_peak3.set_index(['date','station_id','flow'])['count'].reindex(index3,fill_value=0).unstack('flow').reset_index()
assert not panel3[['P','R']].isna().any().any()
means3=panel3.groupby('station_id')[['P','R']].mean().reset_index()
station_stats3=selection_stations3[['station_id','name','region','lat','lon','x','y','days_with_any_event']].merge(means3,on='station_id',validate='one_to_one')
station_stats3['net']=station_stats3.P-station_stats3.R
station_stats3['activity']=station_stats3.P+station_stats3.R
station_stats3['abs_net']=station_stats3.net.abs()
station_stats3['net_pickup']=station_stats3.net.clip(lower=0)
station_stats3['net_return']=(-station_stats3.net).clip(lower=0)

def summarize_regions3(s):
    g=s.groupby('region').agg(station_ids=('station_id','size'),pickups=('P','sum'),returns=('R','sum'),activity=('activity','sum'),sum_abs_station_net=('abs_net','sum'),region_net=('net','sum'),net_pickup_sum=('net_pickup','sum'),net_return_sum=('net_return','sum')).reindex(ORDER3)
    g['activity_per_station']=g.activity/g.station_ids
    g['abs_net_per_station']=g.sum_abs_station_net/g.station_ids
    g['imbalance_share']=g.sum_abs_station_net/g.activity
    g['opposing_flow_scale']=np.minimum(g.net_pickup_sum,g.net_return_sum)
    return g
summary3=summarize_regions3(station_stats3)
# Use +/-1 event per morning as a descriptive near-zero band, not a station filter.
station_stats3['direction']=np.where(station_stats3.net>1,'net pickup',np.where(station_stats3.net< -1,'net return','near balanced'))
directions3=pd.crosstab(station_stats3.region,station_stats3.direction).reindex(ORDER3).fillna(0)
daily3=panel3.merge(station_stats3[['station_id','region']],on='station_id',validate='many_to_one')
daily3['daily_abs_net']=(daily3.P-daily3.R).abs()
summary3['mean_daily_sum_abs_net']=daily3.groupby(['region','date']).daily_abs_net.sum().groupby('region').mean()
station_stats3.to_csv(OUT3/'station_historical_statistics.csv',index=False)
summary3.to_csv(OUT3/'region_comparison_selection_period.csv')
directions3.to_csv(OUT3/'station_direction_counts.csv')
display(summary3.round(1));display(directions3)
assert np.isclose(summary3.pickups.sum(),selection_peak3.loc[selection_peak3.flow=='P','count'].sum()/len(DAYS3))
assert np.isclose(summary3.returns.sum(),selection_peak3.loc[selection_peak3.flow=='R','count'].sum()/len(DAYS3))


### 3.3 Charts: scale, station direction and robustness


In [ ]:
labels3=['Lower Manhattan','Midtown','Upper Manhattan']
fig,axes=plt.subplots(1,2,figsize=(12,4))
summary3[['pickups','returns']].plot.bar(ax=axes[0],color=['#4477aa','#ee9944'],rot=15)
axes[0].set_title('Historical weekday morning pickups / returns');axes[0].set_ylabel('Events per morning');axes[0].set_xlabel('')
summary3.activity_per_station.plot.bar(ax=axes[1],color=[COLORS3[r] for r in ORDER3],rot=15)
axes[1].set_title('Activity per station ID');axes[1].set_ylabel('Events / station / morning');axes[1].set_xlabel('')
plt.tight_layout();fig.savefig(OUT3/'01_activity.png',dpi=160);plt.show()
leader_total3=summary3.activity.idxmax();leader_per3=summary3.activity_per_station.idxmax()
display(Markdown(f'The highest total pickup/return activity is **{leader_total3}**, and the highest activity per station is **{leader_per3}**. High values on both measures cannot be explained by station count alone. These are descriptive selection-period observations, not causal evidence about land use.'))
fig,axes=plt.subplots(1,2,figsize=(12,4))
summary3[['net_pickup_sum','net_return_sum']].plot.bar(ax=axes[0],color=['#bc5090','#008c95'],rot=15)
axes[0].set_title('Station net-flow directions');axes[0].set_ylabel('Mean net events per morning');axes[0].set_xlabel('')
summary3.abs_net_per_station.plot.bar(ax=axes[1],color=[COLORS3[r] for r in ORDER3],rot=15)
axes[1].set_title('Mean absolute station net flow');axes[1].set_ylabel('Net events / station / morning');axes[1].set_xlabel('')
plt.tight_layout();fig.savefig(OUT3/'02_net_flows.png',dpi=160);plt.show()
display(Markdown('The left plot separates net-departure and net-return station flows that cancel in regional totals; the right compares station directionality. Opposing flows do not prove transferable surplus and deficit inventories; midnight stock, capacity and targets are also needed.'))


### 3.4 Sensitivity checks
Keep all selection-period stations in the main comparison. Separately test:

- Assign all near-boundary stations north, assign them south, or exclude them.
- Exclude IDs with coordinate spans above 100 m.
- Keep stations with any-time activity on at least 90% of selection weekdays: at least 39 of 43 days.

The last condition measures observed coverage, not verified operating status. It does not require a morning-peak trip every day.


In [ ]:
variants3={'all_selection_stations':station_stats3.copy()}
near3=review3.loc[review3.near_split,['station_id','nearest_street']].set_index('station_id').nearest_street
for direction in ['north','south']:
    v=station_stats3.copy()
    for sid,street in near3.items():
        region=('Midtown' if street==14 else 'Upper Manhattan') if direction=='north' else ('Lower Manhattan' if street==14 else 'Midtown')
        v.loc[v.station_id==sid,'region']=region
    variants3['boundary_to_'+direction]=v
variants3['exclude_boundary']=station_stats3[~station_stats3.station_id.isin(near3.index)].copy()
uncertain3=review3.loc[review3.position_uncertain,'station_id']
variants3['exclude_uncertain_coordinates']=station_stats3[~station_stats3.station_id.isin(uncertain3)].copy()
coverage_min3=int(np.ceil(.90*len(DAYS3)))
variants3['high_observed_coverage']=station_stats3[station_stats3.days_with_any_event>=coverage_min3].copy()
sensitivity3=pd.concat({name:summarize_regions3(v) for name,v in variants3.items()},names=['scenario','region']).reset_index()
sensitivity3.to_csv(OUT3/'sensitivity_comparison.csv',index=False)
ranking3=sensitivity3.loc[sensitivity3.groupby('scenario').abs_net_per_station.idxmax(),['scenario','region','abs_net_per_station']]
display(ranking3.round(1))
print('Main station count',len(station_stats3),'High-observation-coverage subset',len(variants3['high_observed_coverage']),'Threshold',coverage_min3,' days')
fig,ax=plt.subplots(figsize=(13,4))
sensitivity3.pivot(index='scenario',columns='region',values='abs_net_per_station').reindex(columns=ORDER3).plot.bar(ax=ax,color=[COLORS3[r] for r in ORDER3],rot=20)
ax.legend(loc='upper left',bbox_to_anchor=(1.01,1),title=None)
ax.set_ylabel('Net events / station / morning');ax.set_title('Sensitivity: absolute station net flow');ax.set_xlabel('')
plt.tight_layout();fig.savefig(OUT3/'03_sensitivity.png',dpi=160);plt.show()
stable3=ranking3.region.nunique()==1
leader3=ranking3.region.iloc[0] if stable3 else 'Unstable ranking'
display(Markdown(f'Leader in absolute net flow per station across sensitivity checks: **{leader3}**. Robustness supports the demand-pattern comparison, not a claim of maximum rebalancing benefit. The main analysis keeps all selection-period stations.'))


### Regional selection evidence
Consider totals, activity per station and net flows together without an arbitrary weighted score. Demand patterns support study-area selection, but do not establish optimization benefits.


In [ ]:
# Generate the recommendation from actual results, without hard-coding a preferred region.
leader_activity3=summary3.activity.idxmax()
leader_perstation3=summary3.activity_per_station.idxmax()
leader_imbalance3=summary3.abs_net_per_station.idxmax()
strong3=stable3 and len({leader_activity3,leader_perstation3,leader_imbalance3,leader3})==1
recommendation3=leader_activity3 if strong3 else 'Trade-offs require further review; no automatic selection.'
print('Suggested priority region (subject to team agreement):',recommendation3)
print('Largest opposing-flow scale:',summary3.opposing_flow_scale.idxmax())
print('This does not establish actual rebalancing gains; later stages determine the case and inputs.')
(OUT3/'recommendation.json').write_text(json.dumps({'selection_start':str(SELECT_START3.date()),'selection_end':str(SELECT_END3.date()),'weekday_count':len(DAYS3),'recommended_candidate':recommendation3,'confirmed':False,'optimization_run':False,'prediction_model_trained':False},ensure_ascii=False,indent=2),encoding='utf-8')


### Regional decision: Midtown
Midtown has the highest selection-period total activity, activity per station and absolute net flow per station; the sensitivity checks preserve this result. Lower Manhattan has more balanced opposing flows and remains a useful alternative. These are historical flow statistics, not inventory requirements.


## Step 4: Predict pickups and returns across Midtown
Retain all **190 historical analysis IDs** classified within the main island between 14th and 59th Street. Historical IDs do not establish which physical stations operate on the later case date.

Output one fixed P_i and R_i per station, measured in events/station/weekday morning peak, 06:00–10:00. Predict specific dates first, then average over five weekdays. This stage alone does not produce initial inventories, target stocks or routes.


In [ ]:
# Uncomment the next line if dependencies are missing.
# %pip install pandas numpy matplotlib scikit-learn
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
import sklearn
ROOT4=Path.cwd(); IN4=ROOT4/'outputs_stage2'; OUT4=ROOT4/'outputs_stage4';OUT4.mkdir(exist_ok=True)
TRAIN_END4=pd.Timestamp('2026-08-31'); VAL_END4=pd.Timestamp('2026-09-15'); HISTORY_END4=pd.Timestamp('2026-09-29')
HISTORY_START4=pd.Timestamp('2026-07-02'); FORECAST_START4=pd.Timestamp('2026-10-29')
FUTURE_DATES4=pd.bdate_range(FORECAST_START4,periods=5)
ALPHA4=0.001  # Predefined weak regularization; no validation/test hyperparameter tuning.
MIN_HISTORY_DAYS4=20  # Manual-review flag only; do not exclude stations.
station4=pd.read_csv(IN4/'manhattan_canonical_stations.csv',dtype={'station_id':str})
station4=station4[station4.region=='Midtown'].copy()
assert len(station4)==190 and station4.station_id.is_unique
peak4=pd.read_csv(IN4/'manhattan_daily_peak_counts.csv',dtype={'station_id':str},parse_dates=['date'])
activity4=pd.read_csv(IN4/'manhattan_daily_activity.csv',dtype={'station_id':str},parse_dates=['date'])
print('Scope: all',len(station4),'historical Midtown analysis IDs; scikit-learn',sklearn.__version__)


### 4.1 Observed station-days and chronological splits
A station-day is observed when at least one valid all-day endpoint record exists. If that day has no morning events, the morning count can be zero. A completely unobserved day remains unknown. This avoids assuming closures have zero demand, but misses genuinely operating stations with zero all-day trips. The estimand is therefore demand conditional on activity observation, not unconditional latent demand.

| Stage | Dates in 2026 | Purpose |
|---|---|---|
| Train | July 2–August 31 | Fit both candidates |
| Validate | September 1–15 | Select by MAE, separately for P and R |
| Test | September 16–29 | Refit on train+validation, then report held-out errors |
| Final fit | July 2–September 29 | Generate the five-date static forecast with the selected method |

Fit encoders and date scaling on the relevant training data only. Do not shuffle time or input test outcomes, future realized weather or yesterday's realized trips. This is a retrospective date-based evaluation, not a deployment simulation proving that monthly files were available at each historical prediction date.


In [ ]:
observed4=activity4[activity4.station_id.isin(station4.station_id)&activity4.date.dt.dayofweek.lt(5)&activity4.date.between(HISTORY_START4,HISTORY_END4)][['date','station_id']].drop_duplicates()
wide4=peak4.pivot(index=['date','station_id'],columns='flow',values='count').reset_index()
panel4=observed4.merge(wide4,on=['date','station_id'],how='left',validate='one_to_one')
panel4[['P','R']]=panel4[['P','R']].fillna(0)
panel4['weekday']=panel4.date.dt.dayofweek.astype(str)
panel4['day_index']=(panel4.date-HISTORY_START4).dt.days.astype(float)
panel4=panel4.sort_values(['date','station_id']).reset_index(drop=True)
train4=panel4[panel4.date<=TRAIN_END4].copy()
val4=panel4[(panel4.date>TRAIN_END4)&(panel4.date<=VAL_END4)].copy()
test4=panel4[panel4.date>VAL_END4].copy()
assert train4.date.max()<val4.date.min() and val4.date.max()<test4.date.min()
assert set(panel4.station_id)==set(station4.station_id)
coverage4=panel4.groupby('station_id').agg(history_days=('date','nunique'),first_observed=('date','min'),last_observed=('date','max'),weekdays_observed=('weekday','nunique')).reset_index()
coverage4['training_days']=coverage4.station_id.map(train4.groupby('station_id').size()).fillna(0).astype(int)
coverage4['days_since_last_observation']=(HISTORY_END4-coverage4.last_observed).dt.days
coverage4['short_history']=coverage4.history_days<MIN_HISTORY_DAYS4
coverage4['stale_history']=coverage4.days_since_last_observation>14
coverage4['needs_extra_review']=coverage4.short_history|coverage4.stale_history|coverage4.weekdays_observed.lt(5)
coverage4.to_csv(OUT4/'station_history_coverage.csv',index=False)
panel4.to_csv(OUT4/'observed_station_day_panel.csv',index=False)
split4=pd.DataFrame([{'split':name,'dates':d.date.nunique(),'station_ids':d.station_id.nunique(),'station_days':len(d),'start':d.date.min().date(),'end':d.date.max().date()} for name,d in [('train',train4),('validation',val4),('test',test4)]])
display(split4);display(coverage4[coverage4.needs_extra_review].merge(station4[['station_id','name']],on='station_id'))
print('Unknown station-days (not filled with zero):',len(station4)*len(pd.bdate_range(HISTORY_START4,HISTORY_END4))-len(panel4))


### 4.2 Two simple methods
**Baseline: station-weekday historical means.** Fall back to the station mean when its weekday sample is missing. For unseen stations, fall back to the Midtown weekday mean and flag cold starts separately.

**Candidate: regularized Poisson regression.** Fit one shared pickup model and one return model. Features are station ID, weekday and a date trend. One-hot encode station/weekday and standardize dates within training. The mean is `exp(intercept + station effect + weekday effect + date trend)`. Positive predictions and weak regularization limit unstable estimates; no complex station-specific interactions are added.

Fix `alpha=0.001` without hyperparameter search. Poisson loss estimates a mean; it does not establish equal real-world mean/variance or calibrated probability intervals. [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.PoissonRegressor.html)

Select using validation MAE on previously seen stations only. Report cold starts separately and do not reselect using test outcomes.


In [ ]:
FEATURES4=['station_id','weekday','day_index']
def fit_mean4(d,flow):
    return {'station_weekday':d.groupby(['station_id','weekday'])[flow].mean().to_dict(),
            'station':d.groupby('station_id')[flow].mean().to_dict(),
            'weekday':d.groupby('weekday')[flow].mean().to_dict(),'global':float(d[flow].mean())}

def predict_mean4(model,d):
    values=[];sources=[]
    for sid,dow in zip(d.station_id,d.weekday):
        if (sid,dow) in model['station_weekday']:
            values.append(model['station_weekday'][(sid,dow)]);sources.append('station_weekday')
        elif sid in model['station']:
            values.append(model['station'][sid]);sources.append('station_mean_missing_weekday')
        else:
            values.append(model['weekday'].get(dow,model['global']));sources.append('network_weekday_unseen_station')
    return np.asarray(values),np.asarray(sources)

def fit_poisson4(d,flow):
    transform4=ColumnTransformer([('categorical',OneHotEncoder(handle_unknown='ignore'),['station_id','weekday']),('date',StandardScaler(),['day_index'])])
    model=Pipeline([('features',transform4),('regression',PoissonRegressor(alpha=ALPHA4,max_iter=1500,tol=1e-7))])
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        with threadpool_limits(limits=2):model.fit(d[FEATURES4],d[flow])
    return model

def score_rows4(rows,split,flow,model_name):
    result=[]
    for label,part in [('all_observed',rows),('known_station',rows[rows.known_station]),('unseen_station',rows[~rows.known_station])]:
        if part.empty:continue
        denominator=part.actual.sum()
        result.append({'split':split,'flow':flow,'model':model_name,'cohort':label,'station_days':len(part),'station_ids':part.station_id.nunique(),
                       'MAE':mean_absolute_error(part.actual,part.prediction),
                       'WAPE':np.abs(part.actual-part.prediction).sum()/denominator if denominator else np.nan,
                       'bias':float((part.prediction-part.actual).mean())})
    return result

metrics4=[];prediction_rows4=[];chosen4={};selection_rows4=[]
# Validation stage: the test block below is not touched during this loop.
for flow in ['P','R']:
    mean_model=fit_mean4(train4,flow);poisson_model=fit_poisson4(train4,flow)
    baseline,source=predict_mean4(mean_model,val4)
    predictions={'weekday_mean':baseline,'poisson':poisson_model.predict(val4[FEATURES4])}
    known=val4.station_id.isin(train4.station_id.unique())
    errors={name:mean_absolute_error(val4.loc[known,flow],pred[known.to_numpy()]) for name,pred in predictions.items()}
    chosen4[flow]='poisson' if errors['poisson']<errors['weekday_mean'] else 'weekday_mean'
    selection_rows4.append({'flow':flow,'selected_model':chosen4[flow],'baseline_validation_MAE':errors['weekday_mean'],'poisson_validation_MAE':errors['poisson']})
    for name,pred in predictions.items():
        r=val4[['date','station_id']].copy();r['actual']=val4[flow];r['prediction']=pred;r['known_station']=known;r['flow']=flow;r['model']=name;r['split']='validation'
        metrics4.extend(score_rows4(r,'validation',flow,name));prediction_rows4.append(r)
# Freeze the choices before evaluating the held-out test period.
(OUT4/'frozen_model_selection.json').write_text(json.dumps(chosen4,indent=2),encoding='utf-8')
trainval4=pd.concat([train4,val4],ignore_index=True)
for flow in ['P','R']:
    mean_model=fit_mean4(trainval4,flow);poisson_model=fit_poisson4(trainval4,flow)
    baseline,source=predict_mean4(mean_model,test4)
    predictions={'weekday_mean':baseline,'poisson':poisson_model.predict(test4[FEATURES4])}
    known=test4.station_id.isin(trainval4.station_id.unique())
    for name,pred in predictions.items():
        r=test4[['date','station_id']].copy();r['actual']=test4[flow];r['prediction']=pred;r['known_station']=known;r['flow']=flow;r['model']=name;r['split']='test'
        assert np.isfinite(pred).all() and (pred>=0).all()
        metrics4.extend(score_rows4(r,'test',flow,name));prediction_rows4.append(r)
metrics4=pd.DataFrame(metrics4);prediction4=pd.concat(prediction_rows4,ignore_index=True);selection4=pd.DataFrame(selection_rows4)
metrics4.to_csv(OUT4/'model_metrics.csv',index=False);prediction4.to_csv(OUT4/'validation_test_predictions.csv',index=False);selection4.to_csv(OUT4/'model_selection.csv',index=False)
display(selection4.round(3));display(metrics4[metrics4.cohort=='known_station'].round(3))
display(Markdown('MAE is measured in events/station/weekday morning peak. WAPE is total absolute error divided by total actual events, not accuracy. Bias is mean prediction minus actual; positive values indicate overprediction. Test results do not reselect the method.'))


### 4.3 Validation and held-out test results
The first chart compares candidate methods; the second shows the frozen method on the test period. Daily totals include only observed station-days belonging to previously seen stations. Membership can change by date, so these are not fixed-roster total Midtown demand. Export station-level MAE to avoid hiding errors that cancel in aggregate.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,flow in zip(axes,['P','R']):
    q=metrics4[(metrics4.flow==flow)&(metrics4.cohort=='known_station')].pivot(index='split',columns='model',values='MAE').reindex(['validation','test'])
    q[['weekday_mean','poisson']].plot.bar(ax=ax,rot=0,color=['#718096','#e69f00'])
    ax.set_ylim(0,float(q.max().max())*1.35)
    ax.set_title(('Pickups' if flow=='P' else 'Returns')+' — MAE');ax.set_ylabel('Trips / station / weekday morning');ax.set_xlabel('')
plt.tight_layout();fig.savefig(OUT4/'01_model_comparison.png',dpi=150);plt.show()
display(Markdown('Select methods using validation errors; test data assess generalization only. A more complex method need not improve performance. Use the mean baseline when it wins validation.'))
selected_test4=prediction4[(prediction4.split=='test')&prediction4.known_station&prediction4.apply(lambda r:r.model==chosen4[r.flow],axis=1)].copy()
selected_test4['abs_error']=(selected_test4.prediction-selected_test4.actual).abs()
station_errors4=selected_test4.groupby(['station_id','flow']).agg(test_MAE=('abs_error','mean'),test_days=('date','nunique'),mean_actual=('actual','mean')).reset_index()
station_errors4.to_csv(OUT4/'station_test_errors.csv',index=False)
daily_test4=selected_test4.groupby(['flow','date']).agg(actual=('actual','sum'),prediction=('prediction','sum'),station_ids=('station_id','nunique')).reset_index()
daily_test4.to_csv(OUT4/'daily_test_comparison.csv',index=False)
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
for ax,flow in zip(axes,['P','R']):
    q=daily_test4[daily_test4.flow==flow]
    ax.plot(q.date,q.actual,marker='o',label='Actual matched observations');ax.plot(q.date,q.prediction,marker='o',label='Selected model')
    ax.set_title(('Pickups' if flow=='P' else 'Returns')+' — held-out weekdays');ax.set_ylabel('Trips');ax.legend()
fig.autofmt_xdate();plt.tight_layout();fig.savefig(OUT4/'02_test_actual_prediction.png',dpi=150);plt.show()
display(Markdown('The curves show test-period variation and systematic over/underprediction. Similar totals do not imply accurate station forecasts; also examine station MAE. Weather, events and closures can still cause errors.'))


### 4.4 Final static demand: two values per station
After model selection, refit on all usable history and predict October 29/30 and November 2/3/4. Average each flow separately over these five dates to obtain P_i and R_i.

Poisson extrapolates a date trend by approximately one month; it is not a weather or seasonality forecast. The baseline assumes weekday patterns persist. Retain estimates from both candidates for comparison, but use validation to determine the selected value.

Keep all 190 IDs. Flag fewer than 20 historical weekdays, no observation in the last 14 historical days, or missing weekday coverage for extra review. These are transparent review rules, not confidence guarantees. Future operating status remains unverified; these conditional demand estimates are not yet complete MILP inputs.


In [ ]:
future4=pd.MultiIndex.from_product([FUTURE_DATES4,station4.station_id],names=['date','station_id']).to_frame(index=False)
future4['weekday']=future4.date.dt.dayofweek.astype(str);future4['day_index']=(future4.date-HISTORY_START4).dt.days.astype(float)
final_daily4=[];coefficients4=[];final_models4={}
for flow in ['P','R']:
    mean_model=fit_mean4(panel4,flow);poisson_model=fit_poisson4(panel4,flow);final_models4[flow]=(mean_model,poisson_model)
    baseline,source=predict_mean4(mean_model,future4)
    q=future4[['date','station_id','weekday']].copy();q['flow']=flow;q['weekday_mean']=baseline;q['baseline_source']=source
    q['poisson']=poisson_model.predict(future4[FEATURES4]);q['selected_model']=chosen4[flow];q['prediction']=q[chosen4[flow]]
    assert np.isfinite(q[['weekday_mean','poisson','prediction']].to_numpy()).all() and q[['weekday_mean','poisson','prediction']].ge(0).all().all()
    final_daily4.append(q)
    names=poisson_model.named_steps['features'].get_feature_names_out();coef=poisson_model.named_steps['regression'].coef_
    coefficients4.append(pd.DataFrame({'flow':flow,'feature':names,'coefficient':coef}))
final_daily4=pd.concat(final_daily4,ignore_index=True)
final_daily4.to_csv(OUT4/'five_weekday_predictions.csv',index=False)
pd.concat(coefficients4).to_csv(OUT4/'poisson_coefficients.csv',index=False)
static4=final_daily4.groupby(['station_id','flow']).prediction.mean().unstack('flow').rename(columns={'P':'P_i','R':'R_i'}).reset_index()
static4=station4[['station_id','name','region','lat','lon']].merge(static4,on='station_id',validate='one_to_one').merge(coverage4,on='station_id',validate='one_to_one')
for flow in ['P','R']:
    candidates=final_daily4[final_daily4.flow==flow].groupby('station_id')[['weekday_mean','poisson']].mean()
    static4[f'{flow}_baseline']=static4.station_id.map(candidates.weekday_mean)
    static4[f'{flow}_poisson']=static4.station_id.map(candidates.poisson)
    static4[f'{flow}_selected_model']=chosen4[flow]
static4['demand_unit']='events/station/weekday morning peak'
static4['service_window']='06:00–10:00 America/New_York'
static4['forecast_start']=str(FUTURE_DATES4.min().date());static4['forecast_end']=str(FUTURE_DATES4.max().date())
static4['forecast_weekdays']=5;static4['holidays_excluded']=False
static4['future_operating_status_verified']=False
static4['scenario_assumption']='conditional_on_station_operating; completed-trip demand proxy'
static4['quality_note']=np.where(static4.needs_extra_review,'Short/stale history or incomplete weekday coverage: extra review required.','More complete history; future operating status remains unverified.')
assert len(static4)==190 and static4.station_id.is_unique and static4[['P_i','R_i']].notna().all().all()
static4.to_csv(OUT4/'midtown_static_demand.csv',index=False,encoding='utf-8-sig')
display(static4[['station_id','name','P_i','R_i','history_days','last_observed','quality_note']].round(1).head(15))
print('Exported all',len(static4),'IDs; extra review:',int(static4.needs_extra_review.sum()))
print('Forecast dates:',[str(d.date()) for d in FUTURE_DATES4])
# Use a station with median training activity; do not choose examples using test performance.
train_activity4=train4.groupby('station_id')[['P','R']].mean().sum(axis=1)
example_id4=(train_activity4-train_activity4.median()).abs().idxmin()
example4=final_daily4[final_daily4.station_id==example_id4][['date','flow','weekday_mean','poisson','selected_model','prediction','baseline_source']]
print('Example station for tracing values:',example_id4,station4.set_index('station_id').loc[example_id4,'name'])
display(example4.round(2));example4.to_csv(OUT4/'example_station_calculation.csv',index=False)
display(Markdown('Average the example station prediction column over five dates separately for P and R to obtain P_i and R_i. Baseline daily estimates use the corresponding station-weekday mean; Poisson estimates use the exported station, weekday and date coefficients. Decimals represent expected counts, not decimal-level precision.'))


### 4.5 Demand patterns across stations
The x-axis is estimated pickups and the y-axis estimated returns. Above the diagonal means more returns; below means more pickups. Neither direction establishes surplus or shortage without initial stock, capacity and a target.


In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
for flag,label,color in [(False,'More history; status unverified','#2878a5'),(True,'Extra history/status review','#d95f02')]:
    d=static4[static4.needs_extra_review==flag]
    ax.scatter(d.P_i,d.R_i,s=26,alpha=.7,label=label,color=color)
limit=max(static4.P_i.max(),static4.R_i.max())*1.05
ax.plot([0,limit],[0,limit],'k--',lw=1,label='P = R');ax.set_xlim(0,limit);ax.set_ylim(0,limit)
ax.set_xlabel('Predicted pickups / weekday morning');ax.set_ylabel('Predicted returns / weekday morning');ax.set_title('Midtown: all 190 historical analysis IDs');ax.legend(fontsize=8)
plt.tight_layout();fig.savefig(OUT4/'03_static_demand.png',dpi=150);plt.show()
display(Markdown('Each point is a retained historical analysis ID. Orange marks extra review, not exclusion. The plot checks demand scale and pickup/return differences; it does not establish truck tasks.'))
selected_metrics4=metrics4[(metrics4.split=='test')&(metrics4.cohort=='known_station')&metrics4.apply(lambda r:r.model==chosen4[r.flow],axis=1)]
display(selected_metrics4[['flow','model','MAE','WAPE','bias','station_ids','station_days']].round(3))
summary4={'region':'Midtown','historical_station_ids':len(static4),'selected_models':chosen4,'forecast_dates':[str(d.date()) for d in FUTURE_DATES4],
          'extra_review_station_ids':static4.loc[static4.needs_extra_review,'station_id'].tolist(),'test_is_retrospective':True,'future_status_verified':False,'inventory_parameters_created':False,'optimization_run':False}
(OUT4/'stage4_summary.json').write_text(json.dumps(summary4,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT4/'environment.json').write_text(json.dumps({'sklearn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__},indent=2),encoding='utf-8')
print('Step 4 complete: demand estimates are available, but I0, C, I_star and G are not yet prepared; MILP inputs remain incomplete.')


### Interpreting Step 4
Validation pickup MAE is **12.407** for Poisson versus **12.543** for the baseline, an improvement of about **1.1%**. For returns, the baseline MAE **15.554** is slightly better than Poisson **15.720**. Freeze **Poisson for P and weekday means for R**.

| Flow | Selected method | Test MAE | Test WAPE |
|---|---|---:|---:|
| Pickups P | Poisson | 11.910 | 22.1% |
| Returns R | Weekday mean | 15.027 | 21.8% |

Test evaluation covers 183 previously seen IDs and 1,710 observed station-days. MAE units are events/station/weekday morning peak. Poisson return test MAE is lower at 13.965, showing that the small validation difference is unstable. Do not switch methods using the same test set and still call it independent; a revised method needs a new evaluation window.

All **190 historical IDs** remain in the forecast output, with **7** requiring extra history review. One test-period cold-start ID is reported separately. The evaluation count of 183 is not a reduction in study scope.

Use `outputs_stage4/midtown_static_demand.csv` for conditional P_i/R_i estimates with dates, units, method and quality flags. WAPE is not accuracy. September performance does not verify late-October predictions or current station operations. Before optimization, add case-date station status, capacity and midnight inventory.


In [ ]:
# Reconstruct a Poisson prediction from feature contributions without refitting.
example_row4=future4[(future4.station_id==example_id4)&(future4.date==FUTURE_DATES4[0])]
model4=final_models4['P'][1]
x4=model4.named_steps['features'].transform(example_row4[FEATURES4])
x4=x4.toarray().ravel() if hasattr(x4,'toarray') else np.asarray(x4).ravel()
coef4=model4.named_steps['regression'].coef_;intercept4=float(model4.named_steps['regression'].intercept_)
parts4=pd.DataFrame({'feature':model4.named_steps['features'].get_feature_names_out(),'encoded_value':x4,'coefficient':coef4,'log_mean_contribution':x4*coef4})
parts4=parts4[parts4.encoded_value!=0].copy()
reconstructed4=float(np.exp(intercept4+parts4.log_mean_contribution.sum()))
expected4=float(model4.predict(example_row4[FEATURES4])[0])
assert np.isclose(reconstructed4,expected4)
parts4.to_csv(OUT4/'example_poisson_contributions.csv',index=False)
print('Example station',example_id4,'Date',FUTURE_DATES4[0].date(),'Poisson intercept',round(intercept4,4))
display(parts4.round(4))
print('exp(intercept + sum of contributions above) =',round(reconstructed4,2),'pickup events')


### Output location
Step 4 exports to `outputs_stage4/`. The next section prepares a same-day inventory case; keep the five-day average and case-date predictions distinct.


## Step 5: Midnight inventory and draft parameters
Use midnight inventory on October 28, 2026, New York time, with forecasts for that day's 06:00–10:00 peak. Reuse the models fitted through September 29 and change only the forecast date. Keep historical Midtown IDs and audit additions from the metadata roster. Current metadata supports retrospective matching, not proof of a complete historical roster.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import re
from decimal import Decimal

CASE_DATE5=pd.Timestamp('2026-10-28')
MIDNIGHT5=CASE_DATE5.tz_localize('America/New_York')
TARGET_TS5=int(MIDNIGHT5.timestamp())
OUT5=Path.cwd()/'outputs_stage5'; OUT5.mkdir(exist_ok=True)
CACHE5=Path.cwd()/'data/stage5'; CACHE5.mkdir(parents=True,exist_ok=True)
ARCHIVE5=CACHE5/str(CASE_DATE5.date()); ARCHIVE5.mkdir(exist_ok=True)
INFO_URL5='https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json'
ARCHIVE_URL5='https://ctbk-gbfs-api.hccs-ctbk.workers.dev/api/stations'
# Reuse the fixed snapshot; do not silently refresh it to a different date.
info_path5=CACHE5/'station_information_snapshot.json'
if not info_path5.exists():
    r=requests.get(INFO_URL5,timeout=30);r.raise_for_status();payload=r.json()
    payload['_retrieved_at_utc']=datetime.now(timezone.utc).isoformat()
    info_path5.write_text(json.dumps(payload),encoding='utf-8')
info5=json.loads(info_path5.read_text())
meta5=pd.DataFrame(info5['data']['stations']).rename(columns={'station_id':'gbfs_uuid','short_name':'gbfs_short_name','name':'gbfs_name','lat':'gbfs_lat','lon':'gbfs_lon','capacity':'physical_capacity_snapshot'})
meta5['gbfs_short_name']=meta5.gbfs_short_name.astype(str)
def id_key5(value):
    value=str(value)
    return format(Decimal(value).normalize(),'f') if re.fullmatch(r'[0-9]+(?:\.[0-9]+)?',value) else value
meta5['id_key']=meta5.gbfs_short_name.map(id_key5)
x5,y5=project(meta5.gbfs_lon.to_numpy(),meta5.gbfs_lat.to_numpy())
meta5['in_midtown_snapshot']=[island.covers(Point(x,y)) for x,y in zip(x5,y5)]
meta5['in_midtown_snapshot'] &= (y5>=boundary_y(x5,14))&(y5<boundary_y(x5,59))
meta5['near_split_30m']=[min(boundaries[14].distance(Point(x,y)),boundaries[59].distance(Point(x,y)))<=30 for x,y in zip(x5,y5)]
# Do not arbitrarily choose a UUID when mappings are duplicated.
meta5['duplicate_key']=meta5.id_key.duplicated(keep=False)
assert not meta5.loc[meta5.in_midtown_snapshot,'duplicate_key'].any(), 'Duplicate ID mapping in the Midtown roster; review before continuing.'
meta5.to_csv(OUT5/'station_metadata_audit.csv',index=False)
hist5=static4[['station_id','name','region','lat','lon','history_days','needs_extra_review']].copy()
hist5['id_key']=hist5.station_id.map(id_key5);hist5['has_historical_prediction']=True
keep5=['id_key','gbfs_uuid','gbfs_short_name','gbfs_name','gbfs_lat','gbfs_lon','physical_capacity_snapshot','in_midtown_snapshot','near_split_30m']
roster5=hist5.merge(meta5.loc[~meta5.duplicate_key,keep5],on='id_key',how='left',validate='one_to_one')
new5=meta5[meta5.in_midtown_snapshot&~meta5.id_key.isin(hist5.id_key)].copy()
new5=new5[~new5.duplicate_key].copy()
new5['station_id']=new5.gbfs_short_name;new5['name']=new5.gbfs_name;new5['region']='Midtown'
new5['lat']=new5.gbfs_lat;new5['lon']=new5.gbfs_lon
new5['history_days']=0;new5['needs_extra_review']=True;new5['has_historical_prediction']=False
roster5=pd.concat([roster5,new5[roster5.columns]],ignore_index=True)
roster5['mapping_distance_m']=np.hypot((roster5.lat-roster5.gbfs_lat)*111195,(roster5.lon-roster5.gbfs_lon)*111195*np.cos(np.deg2rad(roster5.lat)))
# Normalize whitespace and the verified Central Park South/S variant; preserve ID suffixes.
def name_key5(value):
    return re.sub(r'\s+',' ',str(value).strip().casefold()).replace('central park south','central park s')
roster5['name_matches']=roster5.name.map(name_key5).eq(roster5.gbfs_name.map(name_key5))
roster5['mapping_ok']=roster5.gbfs_uuid.notna()&roster5.mapping_distance_m.le(50)&roster5.name_matches&roster5.in_midtown_snapshot.fillna(False).astype(bool)
roster5['metadata_snapshot_time']=pd.to_datetime(info5['last_updated'],unit='s',utc=True).tz_convert('America/New_York').isoformat()
roster5.to_csv(OUT5/'midtown_station_roster.csv',index=False,encoding='utf-8-sig')
print('Historical IDs:',len(hist5),'; Midtown metadata IDs:',int(meta5.in_midtown_snapshot.sum()),'; union review roster:',len(roster5))
display(roster5[~roster5.mapping_ok|~roster5.has_historical_prediction][['station_id','name','gbfs_name','has_historical_prediction','mapping_ok']])


### 5.1 Read real midnight observations and retain missingness reasons
Query the 15 minutes before midnight. Select the last observation whose `ts`, `polled_at` and `last_reported` are not later than midnight. Require observation/poll age <=300 seconds and station-report age <=900 seconds, plus installed, renting and returning status. These quality thresholds do not guarantee an exact midnight count.

I0 is available bikes, including e-bikes once. Effective capacity is available bikes plus available docks, assuming unavailable bikes/docks do not recover during this case. Keep physical capacity from the metadata snapshot separately; the two measures are not interchangeable.

Sources: [ctbk](https://ctbk.dev/stations), [official station information](https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json), [GBFS field reference](https://gbfs.org/documentation/reference/). Retain snapshots, responses and request logs.

Normalize whitespace and the verified Central Park South/S variant only. Match IDs and require coordinates within 50 m. If effective capacity exceeds the metadata physical capacity, block target generation pending review. Passing this check does not prove historical capacity was unchanged.


In [ ]:
def fetch_midnight5(uuid):
    path=ARCHIVE5/f'{uuid}.json'
    url=f'{ARCHIVE_URL5}/{uuid}/range'
    params={'from':TARGET_TS5-900,'to':TARGET_TS5}
    if path.exists():
        return uuid,json.loads(path.read_text()),'cached'
    try:
        response=requests.get(url,params=params,timeout=25);response.raise_for_status()
        payload=response.json()
        if not isinstance(payload.get('rows'),list):raise ValueError('archive rows missing')
        payload['_request_url']=response.url;payload['_retrieved_at_utc']=datetime.now(timezone.utc).isoformat()
        path.write_text(json.dumps(payload),encoding='utf-8')
        return uuid,payload,'downloaded'
    except Exception as error:
        return uuid,None,type(error).__name__+': '+str(error)[:160]

archives5={};request_log5=[]
uuids5=roster5.gbfs_uuid.dropna().unique().tolist()
with ThreadPoolExecutor(max_workers=6) as pool:
    jobs=[pool.submit(fetch_midnight5,uuid) for uuid in uuids5]
    for completed,job in enumerate(as_completed(jobs),1):
        uuid,payload,status=job.result();archives5[uuid]=payload
        request_log5.append({'gbfs_uuid':uuid,'result':status,'rows':len(payload['rows']) if payload else 0})
        if completed%40==0:print('Checked',completed,'/',len(jobs),flush=True)
pd.DataFrame(request_log5).to_csv(OUT5/'archive_request_log.csv',index=False)

records5=[]
for row in roster5.itertuples():
    rec={'station_id':row.station_id,'inventory_status':'unmatched_station','I0':np.nan,'C_effective':np.nan}
    payload=archives5.get(row.gbfs_uuid)
    if pd.notna(row.gbfs_uuid):
        rec['inventory_status']='request_failed' if payload is None else 'no_prior_snapshot'
    if payload is not None:
        rec['archive_capacity_reference']=payload.get('capacity') # Unverified timestamp context; not proof of midnight capacity.
        d=pd.DataFrame(payload['rows'])
        if not d.empty:
            required=['ts','polled_at','last_reported','num_bikes_available','num_docks_available','is_installed','is_renting','is_returning']
            if not set(required).issubset(d.columns):
                rec['inventory_status']='missing_required_field'
            else:
                d[required]=d[required].apply(pd.to_numeric,errors='coerce')
                eligible=d[required[:3]].notna().all(axis=1)&d[required[:3]].le(TARGET_TS5).all(axis=1)
                if 'station_id' in d:eligible &= d.station_id.eq(row.gbfs_uuid)
                d=d[eligible].sort_values(['ts','polled_at'])
                if len(d):
                    s=d.iloc[-1]
                    rec.update({k:s.get(k,np.nan) for k in required+['num_ebikes_available','num_bikes_disabled','num_docks_disabled']})
                    rec['snapshot_age_seconds']=TARGET_TS5-s.ts
                    rec['polled_age_seconds']=TARGET_TS5-s.polled_at
                    rec['report_age_seconds']=TARGET_TS5-s.last_reported
                    counts=np.array([s.num_bikes_available,s.num_docks_available],dtype=float)
                    fresh=(rec['snapshot_age_seconds']<=300 and rec['polled_age_seconds']<=300 and rec['report_age_seconds']<=900)
                    valid=np.isfinite(counts).all() and (counts>=0).all() and (counts==np.floor(counts)).all()
                    operating=all(s[k]==1 for k in ['is_installed','is_renting','is_returning'])
                    rec['inventory_status']='valid' if fresh and valid and operating else ('stale_snapshot' if not fresh else 'invalid_counts' if not valid else 'not_operating')
                    if rec['inventory_status']=='valid':
                        rec['I0']=int(s.num_bikes_available);rec['C_effective']=int(counts.sum())
                        if rec['C_effective']==0:rec['inventory_status']='zero_effective_capacity'
    records5.append(rec)
inventory5=pd.DataFrame(records5)
inventory5['case_midnight']=MIDNIGHT5.isoformat()
inventory5.to_csv(OUT5/'midnight_inventory_audit.csv',index=False,encoding='utf-8-sig')
display(inventory5.inventory_status.value_counts().rename_axis('inventory_status').to_frame('station_ids'))


### 5.2 Same-day demand and the original target rule
Apply the fitted models to Wednesday, October 28 for P_case and R_case, in events/station/morning peak. Training stops September 29, creating a one-month extrapolation gap. No October 28 trip outcomes are available here for same-day evaluation. Stations without historical models retain missing demand.

This historical draft uses:

$$I_i^*=\min(C_i,\max(0,\operatorname{round}(C_i/2+(P_i-R_i)/2))),\qquad G_i=I_i^*-I_i^0.$$

Use effective capacity and NumPy ties-to-even rounding. Positive G means delivery relative to the target; negative G means pickup. This heuristic is not an optimal stock level or a service guarantee, and net flow hides within-peak event order. **Step 6's supplement supersedes this rule for routing.**

Ignore natural trips and changes in unavailable bikes/docks during 00:00–06:00. Midnight is the approximate initial state and the target applies at 06:00. Preserve this draft separately from Step 4's five-date averages. Fields map as follows: P_case/R_case to case-day forecasts, I0 to initial available bikes, C_effective to case capacity, I_target to the draft target, and G_target_minus_initial to target minus initial inventory.


In [ ]:
case_features5=station4[['station_id']].copy()
case_features5['date']=CASE_DATE5
case_features5['weekday']=str(CASE_DATE5.dayofweek)
case_features5['day_index']=float((CASE_DATE5-HISTORY_START4).days)
assert CASE_DATE5>HISTORY_END4
for flow in ['P','R']:
    mean_model,poisson_model=final_models4[flow]
    case_features5[flow+'_case']=(poisson_model.predict(case_features5[FEATURES4]) if chosen4[flow]=='poisson' else predict_mean4(mean_model,case_features5)[0])
    case_features5[flow+'_model']=chosen4[flow]
case_features5.to_csv(OUT5/'case_day_demand.csv',index=False)
params5=roster5.merge(inventory5,on='station_id',validate='one_to_one').merge(case_features5[['station_id','P_case','R_case','P_model','R_model']],on='station_id',how='left',validate='one_to_one')
params5['capacity_exceeds_snapshot']=params5.C_effective.gt(params5.physical_capacity_snapshot)
params5['data_complete']=~params5.capacity_exceeds_snapshot&params5.mapping_ok&params5.inventory_status.eq('valid')&params5[['P_case','R_case']].notna().all(axis=1)
params5['I_target']=np.nan;params5['G_target_minus_initial']=np.nan
ok5=params5.data_complete
params5.loc[ok5,'I_target']=np.minimum(params5.loc[ok5,'C_effective'],np.maximum(0,np.rint((params5.loc[ok5,'C_effective']+params5.loc[ok5,'P_case']-params5.loc[ok5,'R_case'])/2)))
params5.loc[ok5,'G_target_minus_initial']=params5.loc[ok5,'I_target']-params5.loc[ok5,'I0']
# Computable does not imply verified; retain sparse/stale history flags separately.
params5['needs_history_review']=params5.needs_extra_review.astype(bool)
params5['candidate_without_history_flag']=params5.data_complete&~params5.needs_history_review
params5['case_date']=str(CASE_DATE5.date());params5['demand_unit']='events/station/06:00-10:00'
params5['inventory_unit']='bikes available for rental';params5['capacity_definition']='available_bikes_plus_available_docks_at_snapshot'
params5['model_assumptions']='fixed_available_capacity; overnight_natural_flows_ignored; net_flow_heuristic'
params5['ready_for_optimization']=False
params5.to_csv(OUT5/'midtown_parameter_draft.csv',index=False,encoding='utf-8-sig')
review5=params5[~params5.data_complete|params5.needs_history_review].copy()
review5.to_csv(OUT5/'stations_requiring_review.csv',index=False,encoding='utf-8-sig')
print('Full review roster retained:',len(params5),'; computable parameters:',int(ok5.sum()),'; computable without history flags:',int(params5.candidate_without_history_flag.sum()))
display(params5[['station_id','name','P_case','R_case','I0','C_effective','I_target','G_target_minus_initial','inventory_status','needs_history_review']].round(1).head(12))
display(review5[['station_id','name','has_historical_prediction','inventory_status','mapping_ok','needs_history_review']])


### 5.3 Sanity checks
Require integer inventory/targets within capacity and nonnegative forecasts. Invalid rows do not enter the computable subset. Empty departure and return without external supply imply fleet conservation; independently generated station targets may not be simultaneously attainable. A soft penalty permits deviations but does not turn every target gap into a mandatory transfer.

The chart shows directions and totals only for complete-data stations, not missing inventory as zero or a complete Midtown solution. Resolve identity, capacity and overnight-flow assumptions before finalizing the model case.

These tables support Milestone 4's parameter definitions, real-data checks and sanity checks. They do not alone supply the final algebraic model, working solver, candidate solution or report.


In [ ]:
assert params5.station_id.is_unique and len(params5)>=190
q5=params5[params5.data_complete].copy()
assert q5[['P_case','R_case','I0','C_effective','I_target']].notna().all().all()
assert q5[['P_case','R_case','I0','C_effective','I_target']].ge(0).all().all()
assert q5.I0.le(q5.C_effective).all() and q5.I_target.le(q5.C_effective).all()
assert np.equal(q5[['I0','C_effective','I_target']],np.floor(q5[['I0','C_effective','I_target']])).all().all()
assert np.allclose(q5.G_target_minus_initial,q5.I_target-q5.I0)
assert params5.loc[~params5.data_complete,['I_target','G_target_minus_initial']].isna().all().all()
assert q5[['ts','polled_at','last_reported']].le(TARGET_TS5).all().all()
summary5={'case_date':str(CASE_DATE5.date()),'historical_ids':len(hist5),'metadata_midtown_ids':int(meta5.in_midtown_snapshot.sum()),'union_roster_ids':len(params5),'data_complete_ids':int(params5.data_complete.sum()),'no_history_flag_ids':int(params5.candidate_without_history_flag.sum()),'review_ids':len(review5),'initial_bikes_in_complete_subset':int(q5.I0.sum()),'target_bikes_in_complete_subset':int(q5.I_target.sum()),'net_target_gap_in_complete_subset':int(q5.G_target_minus_initial.sum()),'full_region_verified':False,'optimization_run':False,'history_end':str(HISTORY_END4.date())}
(OUT5/'stage5_summary.json').write_text(json.dumps(summary5,ensure_ascii=False,indent=2),encoding='utf-8')
fig,axes=plt.subplots(1,2,figsize=(11,4))
counts5=pd.Series({'G > 0':q5.G_target_minus_initial.gt(0).sum(),'G = 0':q5.G_target_minus_initial.eq(0).sum(),'G < 0':q5.G_target_minus_initial.lt(0).sum()})
axes[0].bar(counts5.index,counts5.values,color=['#d95f02','#8b949e','#2878a5']);axes[0].set_ylabel('Station IDs');axes[0].set_title('Direction relative to heuristic target')
for i,v in enumerate(counts5.values):axes[0].text(i,v+1,str(v),ha='center')
axes[1].bar(['Initial I0','Heuristic target'],[q5.I0.sum(),q5.I_target.sum()],color=['#2878a5','#d95f02']);axes[1].set_ylabel('Available bikes');axes[1].set_title('Totals: complete-data subset only')
for i,v in enumerate([q5.I0.sum(),q5.I_target.sum()]):axes[1].text(i,v+20,f'{v:.0f}',ha='center')
for ax in axes:ax.set_ylim(0,ax.get_ylim()[1]*1.15)
fig.suptitle(f'Midtown case {CASE_DATE5.date()} — {len(q5)} calculable IDs, not a routing solution')
plt.tight_layout();fig.savefig(OUT5/'parameter_sanity_check.png',dpi=150);plt.show()
display(Markdown(f'The left plot has {int(counts5.iloc[0])} IDs below the heuristic target and {int(counts5.iloc[2])} above it. These are directions relative to a target, not verified shortages/overflows. For the {len(q5)} computable IDs, target total minus initial available bikes is **{int(q5.G_target_minus_initial.sum()):+,} bikes**. Exact attainment in a closed region requires a zero aggregate gap. Unresolved Midtown records remain; this subset is not the full region.'))
print(json.dumps(summary5,ensure_ascii=False,indent=2))


## Step 6: Retained stations and final inputs
Exclude disputed identity, inventory, capacity or history records without random imputation. Use October 28 midnight inventory, same-day 06:00–10:00 forecasts and available capacity. The first cell reproduces the old half-capacity target for traceability; **the supplement below replaces it with the fleet-conserving targets used in Step 8**. Results represent only the retained audited Midtown subset.


In [ ]:
from pathlib import Path
import csv, json, math
OUT6=Path('outputs_stage6'); OUT6.mkdir(exist_ok=True)
with open('outputs_stage5/midtown_parameter_draft.csv',encoding='utf-8-sig',newline='') as f:
    rows6=list(csv.DictReader(f))
def flag6(row,key):
    return str(row.get(key,'')).lower()=='true'
def number6(row,key):
    try:return float(row[key])
    except (ValueError,KeyError):return float('nan')
accepted6=[];excluded6=[]
for r in rows6:
    reasons=[]
    if not flag6(r,'mapping_ok'):reasons.append('Station identity or location unverified')
    if not flag6(r,'has_historical_prediction'):reasons.append('No historical demand model')
    if r['inventory_status']!='valid':reasons.append('Inventory status: '+r['inventory_status'])
    if flag6(r,'capacity_exceeds_snapshot'):reasons.append('Effective capacity exceeds metadata capacity')
    if flag6(r,'needs_history_review'):reasons.append('Short/stale history or incomplete weekday coverage')
    vals={k:number6(r,k) for k in ['I0','C_effective','P_case','R_case','lat','lon']}
    if not all(math.isfinite(v) for v in vals.values()):reasons.append('Missing or non-finite key parameters')
    if not reasons:
        C,I0,P,R=[vals[k] for k in ['C_effective','I0','P_case','R_case']]
        assert C>0 and 0<=I0<=C and P>=0 and R>=0
        assert C==int(C) and I0==int(I0)
        target=min(int(C),max(0,round((C+P-R)/2)))
        assert target==number6(r,'I_target')
        assert r['case_date']=='2026-10-28'
        accepted6.append({'id':r['station_id'],'name':r['name'],'lat':vals['lat'],'lon':vals['lon'],
          'C':int(C),'I0':int(I0),'P':P,'R':R,'I_star':target,'G':target-int(I0),
          'case_date':r['case_date'],'demand_window':'06:00-10:00 America/New_York',
          'capacity_basis':'midnight available bikes + available docks'})
    else:
        excluded6.append({'station_id':r['station_id'],'name':r['name'],'exclusion_reasons':'; '.join(reasons)})
assert len({r['station_id'] for r in rows6})==len(rows6)
assert len(accepted6)+len(excluded6)==len(rows6)
assert not ({r['id'] for r in accepted6}&{r['station_id'] for r in excluded6})
for filename,records in [('final_station_parameters.csv',accepted6),('excluded_stations.csv',excluded6)]:
    with (OUT6/filename).open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
dataset6={'stations':accepted6}
(OUT6/'model_station_inputs.json').write_text(json.dumps(dataset6,indent=2),encoding='utf-8')
summary6={'review_roster':len(rows6),'included':len(accepted6),'excluded':len(excluded6),
 'initial_bikes':sum(r['I0'] for r in accepted6),'target_bikes':sum(r['I_star'] for r in accepted6),
 'no_rebalancing_mismatch':sum(abs(r['G']) for r in accepted6),
 'net_target_gap':sum(r['G'] for r in accepted6),'optimization_run':False}
(OUT6/'stage6_summary.json').write_text(json.dumps(summary6,indent=2))
print(json.dumps(summary6,indent=2))
print('Excluded stations and reasons:')
for r in excluded6:print(r['station_id'],r['name'],'—',r['exclusion_reasons'])


## Step 6 supplement: Fleet-conserving target inventory
The old targets total 4,106 versus 1,978 initial bikes. Allocate those **1,978 bikes across 166 stations**, with integer targets bounded by each effective capacity. Preserve the original forecasts and old targets for reference. The case remains October 28 midnight and the 06:00–10:00 peak.

**Data:** All five September CSV parts contain 4,982,855 rides. Extract 407,789 morning pickup/return events for the 166 stations over September 1–29 weekdays, including holidays. Replay only station-days with all-day activity evidence, including observed days with zero peak events; do not treat completely unknown days as zeros. There are 3,389 observed station-days. Compact sequences and provenance in `data/target_inventory/` avoid redownloading the approximately 972 MB raw ZIP.

**Replay:** Historical sequences provide event order. Scale pickup and return events separately by case forecast divided by the corresponding historical mean, so mean replay totals match the forecasts. Try every integer initial stock per station. Equally penalize blocked pickup/return volumes; blocked operations do not change stock and do not reroute to another station. Fractional intermediate stock is a fluid approximation, not physically divisible bikes. Process returns before pickups at identical timestamps as a tie convention. Assume September event ordering approximates October 28 and use fixed case capacity. Completed trips omit already unsuccessful attempts.

**Allocation:** Dynamic programming minimizes mean replay loss subject to exact total-stock conservation. Within a 1e-9 loss tolerance, prefer lower total adjustment from initial inventories. This is neither proportional scaling of old targets nor a forced increase in transfer tasks. Optimality refers to this replay allocation model; no independent service test has been performed.

**Execution:** The next cell overwrites `model_station_inputs.json`, `final_station_parameters.csv` and summaries for Step 8. Back up old targets under `outputs_stage6/previous_targets`. Rerun Step 8; old routes do not correspond to new targets. Step 7 uses unweighted distances, so the existing two depots remain applicable.


In [ ]:
REBUILD_EVENT_CACHE = False  # Set True only to rebuild the cache from a local official September raw ZIP.
if REBUILD_EVENT_CACHE:
    import pandas as pd, json, zipfile, csv, gzip, hashlib
    from pathlib import Path
    out=Path('data/target_inventory');out.mkdir(parents=True,exist_ok=True)
    st=json.load(open('outputs_stage6/model_station_inputs.json'))['stations'];ids={s['id'] for s in st}
    a=pd.read_csv('outputs_stage2/alias_resolution.csv',dtype=str)
    aliases=dict(zip(a.station_id,a.canonical_station_id))
    parts=[];seen=set();total=0;dups=0
    with zipfile.ZipFile('data/raw/202609-citibike-tripdata.zip') as z:
     for name in z.namelist():
      if not name.endswith('.csv') or name.startswith('__MACOSX/'):continue
      print('Reading',name,flush=True)
      with z.open(name) as f:
       for chunk in pd.read_csv(f,usecols=['ride_id','started_at','ended_at','start_station_id','end_station_id'],dtype=str,chunksize=200000):
        total+=len(chunk);chunk=chunk.dropna(subset=['ride_id'])
        dup=chunk.ride_id.duplicated()|chunk.ride_id.isin(seen);dups+=int(dup.sum());seen.update(chunk.ride_id);chunk=chunk.loc[~dup]
        for side,col,delta in [('start','started_at',-1),('end','ended_at',1)]:
         sid=chunk[side+'_station_id'].str.strip();sid=sid.map(aliases).fillna(sid)
         mask=sid.isin(ids)
         t=pd.to_datetime(chunk.loc[mask,col],errors='coerce',format='mixed')
         keep=t.notna()&(t.dt.dayofweek<5)&(t.dt.hour>=6)&(t.dt.hour<10)&(t.dt.normalize().between('2026-09-01','2026-09-29'))
         idx=t.index[keep];times=t.loc[idx]
         parts.append(pd.DataFrame({'station_id':sid.loc[idx],'date':times.dt.strftime('%Y-%m-%d'),
               'time':times.dt.strftime('%H:%M:%S.%f'),'delta':delta}))
    events=pd.concat(parts,ignore_index=True).sort_values(['station_id','date','time','delta'],ascending=[True,True,True,False])
    activity=pd.read_csv('outputs_stage2/manhattan_daily_activity.csv',dtype={'station_id':str},parse_dates=['date'])
    print(activity.columns,flush=True)
    days=activity[activity.station_id.isin(ids)&activity.date.between('2026-09-01','2026-09-29')&(activity.date.dt.dayofweek<5)][['station_id','date']].drop_duplicates()
    days['date']=days.date.dt.strftime('%Y-%m-%d')
    events.to_csv(out/'september_peak_events.csv.gz',index=False,compression='gzip')
    days.to_csv(out/'observed_station_days.csv',index=False)
    manifest={'source':'https://tripdata.s3.amazonaws.com/202609-citibike-tripdata.zip',
     'source_zip_bytes':Path('data/raw/202609-citibike-tripdata.zip').stat().st_size,'rows_read':total,'duplicate_ride_ids':dups,
     'events':len(events),'observed_station_days':len(days),'start':'2026-09-01','end':'2026-09-29',
     'window':'06:00-10:00 America/New_York; Monday-Friday including holidays',
     'equal_timestamp_rule':'returns before pickups (arbitrary tie convention)',
     'event_cache_sha256':hashlib.sha256((out/'september_peak_events.csv.gz').read_bytes()).hexdigest()}
    (out/'source_manifest.json').write_text(json.dumps(manifest,indent=2))
    print(manifest,flush=True)


In [ ]:
# Step 6 supplement: allocate the existing fleet using ordered peak events.
from pathlib import Path
import json, csv, shutil
import numpy as np
import pandas as pd

target_out=Path('outputs_stage6');target_out.mkdir(exist_ok=True)
old_dir=target_out/'previous_targets';old_dir.mkdir(exist_ok=True)
for filename in ['model_station_inputs.json','final_station_parameters.csv','stage6_summary.json']:
    if not (old_dir/filename).exists():shutil.copy2(target_out/filename,old_dir/filename)
original_target_data=json.loads((old_dir/'model_station_inputs.json').read_text())
target_stations=sorted(original_target_data['stations'],key=lambda s:s['id'])
target_events=pd.read_csv('data/target_inventory/september_peak_events.csv.gz',dtype={'station_id':str})
target_days=pd.read_csv('data/target_inventory/observed_station_days.csv',dtype={'station_id':str})
target_source=json.loads(Path('data/target_inventory/source_manifest.json').read_text())
target_events=target_events.sort_values(['station_id','date','time','delta'],ascending=[True,True,True,False])
assert len(target_stations)==166 and len({s['id'] for s in target_stations})==166
assert all(s['case_date']=='2026-10-28' for s in target_stations)
assert target_events.date.max() < '2026-10-28'
station_events={sid:g for sid,g in target_events.groupby('station_id')}
station_dates={sid:list(g.date) for sid,g in target_days.groupby('station_id')}
cost_curves=[];curve_rows=[];coverage_rows=[]
for s in target_stations:
    sid=s['id'];capacity=int(s['C']);dates=station_dates.get(sid,[])
    if not dates:raise ValueError(f'{sid} has no historical weekdays to replay; stop rather than invent sequences.')
    group=station_events.get(sid,pd.DataFrame(columns=['date','delta']))
    mean_p=float((group.delta<0).sum())/len(dates)
    mean_r=float((group.delta>0).sum())/len(dates)
    # Keep case-day P/R forecasts; historical sequences supply order and scaled mean flow totals.
    if (mean_p==0 and s['P']>0) or (mean_r==0 and s['R']>0):
        raise ValueError(f'{sid} has positive predicted flow without historical events in that direction; scaling is unsupported.')
    p_weight=s['P']/mean_p if mean_p else 0
    r_weight=s['R']/mean_r if mean_r else 0
    day_sequences={d:g.delta.to_numpy() for d,g in group.groupby('date')}
    losses=np.zeros(capacity+1)
    for date in dates:
        stock=np.arange(capacity+1,dtype=float)
        for delta in day_sequences.get(date,[]):
            proposed=stock+(r_weight if delta>0 else -p_weight)
            losses+=np.maximum(-proposed,0)+np.maximum(proposed-capacity,0)
            stock=np.clip(proposed,0,capacity)
    losses/=len(dates)
    cost_curves.append(losses)
    for b,value in enumerate(losses):curve_rows.append({'station_id':sid,'initial_bikes':b,'mean_simulated_loss':float(value)})
    coverage_rows.append({'station_id':sid,'observed_weekdays':len(dates),'historical_mean_pickups':mean_p,
        'historical_mean_returns':mean_r,'pickup_event_weight':p_weight,'return_event_weight':r_weight})

# Dynamic programming chooses integer station targets minimizing replay loss with exact conservation.
# Within 1e-9 replay-loss tolerance, prefer less total adjustment from initial stocks.
total_bikes=sum(int(s['I0']) for s in target_stations)
dp=np.full(total_bikes+1,np.inf);dp[0]=0
secondary=np.full(total_bikes+1,np.inf);secondary[0]=0
choices=np.full((len(target_stations),total_bikes+1),-1,dtype=np.int16)
for i,(s,costs) in enumerate(zip(target_stations,cost_curves)):
    next_dp=np.full_like(dp,np.inf);next_secondary=np.full_like(dp,np.inf)
    for bikes,cost in enumerate(costs[:total_bikes+1]):
        candidate=dp[:total_bikes+1-bikes]+cost
        candidate_secondary=secondary[:total_bikes+1-bikes]+abs(bikes-s['I0'])
        old=next_dp[bikes:];old_secondary=next_secondary[bikes:]
        tie=np.isfinite(candidate)&np.isfinite(old)&(np.abs(np.where(np.isfinite(candidate),candidate,0)-np.where(np.isfinite(old),old,0))<=1e-9)
        better=np.isfinite(candidate)&((candidate<old-1e-9)|(tie&(candidate_secondary<old_secondary)))
        old[better]=candidate[better];old_secondary[better]=candidate_secondary[better]
        choices[i,bikes:][better]=bikes
    dp,secondary=next_dp,next_secondary
assert np.isfinite(dp[total_bikes]),'No feasible target allocation under fleet-total and capacity constraints.'
new_targets=[];remaining=total_bikes
for i in range(len(target_stations)-1,-1,-1):
    bikes=int(choices[i,remaining]);assert bikes>=0
    new_targets.append(bikes);remaining-=bikes
new_targets=list(reversed(new_targets));assert remaining==0
comparison=[]
for s,new,costs in zip(target_stations,new_targets,cost_curves):
    previous=int(s['I_star']);s['I_star']=int(new);s['G']=int(new-s['I0'])
    s['target_method']='fixed_fleet_scaled_historical_event_replay_v1'
    comparison.append({'station_id':s['id'],'name':s['name'],'C':s['C'],'I0':s['I0'],'P':s['P'],'R':s['R'],
        'old_target':previous,'new_target':new,'old_G':previous-s['I0'],'new_G':s['G'],
        'loss_at_initial':float(costs[int(s['I0'])]),'loss_at_old_target':float(costs[previous]),'loss_at_new_target':float(costs[new])})
assert sum(s['I_star'] for s in target_stations)==total_bikes
assert sum(s['G'] for s in target_stations)==0
assert all(0<=s['I_star']<=s['C'] for s in target_stations)
new_loss=sum(r['loss_at_new_target'] for r in comparison)
initial_loss=sum(r['loss_at_initial'] for r in comparison)
assert new_loss<=initial_loss+1e-6,'New targets must not perform worse than the feasible initial-stock allocation in this replay model.'
target_summary={'case_date':'2026-10-28','target_method':'fixed_fleet_scaled_historical_event_replay_v1',
    'stations':len(target_stations),'initial_inventory_total':total_bikes,
    'old_target_total':sum(r['old_target'] for r in comparison),'target_inventory_total':sum(new_targets),
    'net_target_gap':0,'surplus_stations':sum(s['G']<0 for s in target_stations),
    'deficit_stations':sum(s['G']>0 for s in target_stations),'balanced_stations':sum(s['G']==0 for s in target_stations),
    'surplus_bikes':sum(max(-s['G'],0) for s in target_stations),'deficit_bikes':sum(max(s['G'],0) for s in target_stations),
    'simulated_mean_loss_initial':initial_loss,'simulated_mean_loss_old_target':sum(r['loss_at_old_target'] for r in comparison),
    'simulated_mean_loss_new_target':new_loss,'historical_sequence_start':target_source['start'],
    'historical_sequence_end':target_source['end'],'event_cache_sha256':target_source['event_cache_sha256'],
    'independent_service_test':False,'routing_optimization_run':False}
def save_target_csv(name,rows):
    with (target_out/name).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
save_target_csv('target_comparison.csv',comparison)
save_target_csv('inventory_service_curves.csv',curve_rows)
save_target_csv('event_replay_coverage.csv',coverage_rows)
save_target_csv('final_station_parameters.csv',target_stations)
dataset6={'stations':target_stations}
(target_out/'model_station_inputs.json').write_text(json.dumps(dataset6,ensure_ascii=False,indent=2))
(target_out/'stage6_summary.json').write_text(json.dumps(target_summary,ensure_ascii=False,indent=2))
(target_out/'new_target_summary.json').write_text(json.dumps(target_summary,ensure_ascii=False,indent=2))
print(json.dumps(target_summary,ensure_ascii=False,indent=2))
print('New targets written to the input files read by Step 8. Old targets are in outputs_stage6/previous_targets.')
print('Rerun Step 8; the old solution.json does not correspond to the new targets.')


## Step 7: Select two depot candidates
Use each retained station coordinate as a candidate location. Enumerate all distinct pairs and minimize the unweighted sum of distance from each station to its nearest candidate: a simplified discrete two-median model.

Use D = 84|delta longitude| + 111|delta latitude| km, rounded to three decimals. This is not road distance and ignores one-way streets. Compare with the optimal one-depot baseline.

Depots are independent routing nodes without extra bike supply, even when colocated with a station. Nearest-depot groups illustrate coverage, not restrictions on truck assignments. Optimality refers only to the candidate-set location objective, not joint route optimization or practical site feasibility.


In [ ]:
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
OUT7=Path('outputs_stage7');OUT7.mkdir(exist_ok=True)
stations7=sorted(dataset6['stations'],key=lambda r:r['id'])
lat7=np.array([s['lat'] for s in stations7]);lon7=np.array([s['lon'] for s in stations7])
D7=np.round(84*np.abs(lon7[:,None]-lon7[None,:])+111*np.abs(lat7[:,None]-lat7[None,:]),3)
assert np.isfinite(D7).all() and np.allclose(D7,D7.T) and (D7>=0).all()
# Sort station IDs and keep the first enumerated tied pair for reproducibility.
pairs7=[]
for a,c in combinations(range(len(stations7)),2):
    pairs7.append((float(np.minimum(D7[:,a],D7[:,c]).sum()),a,c))
best7=min(pairs7);cost7,a7,c7=best7
chosen7=[a7,c7];nearest7=np.argmin(D7[:,chosen7],axis=1)
nearest_distance7=np.min(D7[:,chosen7],axis=1)
one7=int(np.argmin(D7.sum(axis=0)));one_cost7=float(D7[:,one7].sum())
depots7=[]
for k,j in enumerate(chosen7):
    st=stations7[j]
    depots7.append({'truck_id':f'Truck_{k+1}','depot_id':f'depot_Truck_{k+1}',
      'name':f'Depot {k+1} (candidate near '+st['name']+')','lat':st['lat'],'lon':st['lon'],
      'candidate_station_id':st['id'],'nearest_station_count':int((nearest7==k).sum()),
      'site_feasibility_verified':False})
assert len(depots7)==2 and depots7[0]['depot_id']!=depots7[1]['depot_id']
assert np.isclose(nearest_distance7.sum(),cost7)
assert cost7<=one_cost7+1e-9
(OUT7/'selected_depots.json').write_text(json.dumps(depots7,ensure_ascii=False,indent=2))
with (OUT7/'station_depot_distances.csv').open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.writer(f);w.writerow(['station_id','nearest_depot','distance_km','distance_depot_1_km','distance_depot_2_km'])
    for i,st in enumerate(stations7):w.writerow([st['id'],depots7[int(nearest7[i])]['depot_id'],nearest_distance7[i],D7[i,a7],D7[i,c7]])
with (OUT7/'all_candidate_pairs.csv').open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.writer(f);w.writerow(['total_nearest_distance_km','candidate_1','candidate_2'])
    for cost,a,c in sorted(pairs7):w.writerow([cost,stations7[a]['id'],stations7[c]['id']])
summary7={'candidate_count':len(stations7),'pairs_evaluated':len(pairs7),'one_depot_total_km':one_cost7,
 'two_depot_total_km':cost7,'two_depot_mean_km':float(nearest_distance7.mean()),
 'two_depot_max_km':float(nearest_distance7.max()),'reduction_percent':100*(1-cost7/one_cost7),
 'objective':'unweighted_sum_of_nearest_depot_distances','distance':'rounded_coordinate_L1_km',
 'site_feasibility_verified':False,'routing_optimization_run':False}
(OUT7/'stage7_summary.json').write_text(json.dumps(summary7,indent=2))
print('Candidate locations:',len(stations7),'; pairs evaluated:',len(pairs7))
for d in depots7:print(d['depot_id'],d['name'],'coordinates',d['lat'],d['lon'],'nearest-station count',d['nearest_station_count'])
print(f'Best single-depot total distance: {one_cost7:.3f} km; two depots: {cost7:.3f} km')
print(f'Mean nearest-depot distance: {nearest_distance7.mean():.3f} km; maximum: {nearest_distance7.max():.3f} km; reduction vs one depot: {summary7["reduction_percent"]:.1f}%')
fig7,ax7=plt.subplots(figsize=(8,7))
colors7=['#2878a5','#d95f02']
for k,j in enumerate(chosen7):
    mask=nearest7==k
    ax7.scatter(lon7[mask],lat7[mask],s=15,color=colors7[k],label=f'Nearest depot {k+1} ({mask.sum()} stations)')
    ax7.scatter(lon7[j],lat7[j],s=230,marker='*',c='gold',edgecolors='black',zorder=5)
    ax7.annotate(f'Depot {k+1}',(lon7[j],lat7[j]),xytext=(8,8),textcoords='offset points',weight='bold')
ax7.set_aspect(1/np.cos(np.deg2rad(lat7.mean())))
ax7.set_xlabel('Longitude');ax7.set_ylabel('Latitude');ax7.set_title('Midtown: two candidate depots and 166 retained station IDs')
ax7.legend();fig7.tight_layout();fig7.savefig(OUT7/'depot_candidates.png',dpi=150)
plt.show()


Stars mark selected candidate sites; colors indicate the nearest depot, not fixed truck service areas. Improvements in this location metric are not equivalent to reductions in total routed distance or cost. Parking, loading and access remain unverified.

Outputs in `outputs_stage7/` include depot inputs, station distances, all candidate pairs and a summary.


## Step 8: Solve the new-target routing model with Gurobi
Confirm that target and midnight totals match, then run both cells in order. Include all 166 retained stations, two trucks, Q=25, 25 km per truck, alpha=1.5, beta=5 and a 120-second optimization limit. Model building and warm-start construction take additional time. Gurobi is required; license errors do not trigger a HiGHS fallback. Targets remain soft: total-stock consistency does not guarantee that the two trucks can reach every target within operational limits.


In [ ]:
import os
import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Any
def latlon_to_km(lat1,lon1,lat2,lon2):
    return round(84*abs(lon1-lon2)+111*abs(lat1-lat2),3)


def init_gurobi_env(license_path: str = "gurobi.lic") -> gp.Env:
    """Initializes Gurobi Environment using local gurobi.lic license file."""
    abs_lic_path = os.path.abspath(license_path)
    if os.path.exists(abs_lic_path) and not os.environ.get("GRB_LICENSE_FILE"):
        os.environ["GRB_LICENSE_FILE"] = abs_lic_path
        print(f"Loading Gurobi license from: {abs_lic_path}")
    
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 1)
    env.start()
    return env


def solve_citi_bike_rebalancing(
    data: Dict[str, Any],
    depots: List[Dict[str, Any]],
    truck_capacity: int = 25,
    max_distance_w: float = 25.0,
    alpha: float = 1.5,
    beta: float = 5.0,
    license_path: str = "gurobi.lic"
) -> Dict[str, Any]:
    """Builds and solves the overnight rebalancing MILP (Milestone 3) with Subtour Elimination."""
    stations = data['stations']
    I = [s['id'] for s in stations]
    K = [d['truck_id'] for d in depots]
    
    C = {s['id']: s['C'] for s in stations}
    I0 = {s['id']: s['I0'] for s in stations}
    G = {s['id']: s['G'] for s in stations}
    
    depot_map = {d['truck_id']: d for d in depots}
    
    # Distance Matrix D_ij
    D = {}
    for k in K:
        nodes = I + [f"depot_{k}"]
        coords = {s['id']: (s['lat'], s['lon']) for s in stations}
        coords[f"depot_{k}"] = (depot_map[k]['lat'], depot_map[k]['lon'])
        
        for i in nodes:
            for j in nodes:
                if i != j:
                    lat1, lon1 = coords[i]
                    lat2, lon2 = coords[j]
                    D[(i, j, k)] = latlon_to_km(lat1, lon1, lat2, lon2)

    max_capacity = max(C.values()) if len(C) > 0 else 50
    M = 2 * truck_capacity + max_capacity

    env = init_gurobi_env(license_path=license_path)
    model = gp.Model("CitiBike_Midtown_Rebalancing", env=env)
    model.Params.TimeLimit = 120
    model.Params.Threads = 2

    # Decision Variables
    x = {}  # Routing variable x_{ijk} in {0,1}
    z = {}  # Net delivery z_{ik} in Integer
    l = {}  # Load l_{ik} >= 0
    u = {}  # Target shortfall u_i >= 0
    y = {}  # Subtour elimination sequence variable y_{ik} >= 0

    for i in I:
        u[i] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"u_{i}")

    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        
        l[(depot_k, k)] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=0, name=f"l_{depot_k}_{k}")
        
        for i in I:
            z[(i, k)] = model.addVar(vtype=GRB.INTEGER, lb=-truck_capacity, ub=truck_capacity, name=f"z_{i}_{k}")
            l[(i, k)] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=truck_capacity, name=f"l_{i}_{k}")
            y[(i, k)] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=len(I), name=f"y_{i}_{k}")

        for i in nodes_k:
            for j in nodes_k:
                if i != j:
                    x[(i, j, k)] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{k}")

    model.update()

    # Objective Function (Equation 3)
    transport_cost = gp.quicksum(alpha * D[(i, j, k)] * x[(i, j, k)] 
                                for k in K 
                                for i in I + [f"depot_{k}"] 
                                for j in I + [f"depot_{k}"] if i != j)
    penalty_cost = gp.quicksum(beta * u[i] for i in I)
    model.setObjective(transport_cost + penalty_cost, GRB.MINIMIZE)

    # (4) Depot Flow Balance
    for k in K:
        depot_k = f"depot_{k}"
        model.addConstr(
            gp.quicksum(x[(depot_k, j, k)] for j in I) == gp.quicksum(x[(i, depot_k, k)] for i in I),
            name=f"depot_flow_balance_{k}"
        )
        model.addConstr(
            gp.quicksum(x[(depot_k, j, k)] for j in I) <= 1,
            name=f"depot_use_{k}"
        )

    # (5) Route Continuity
    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        for i in nodes_k:
            model.addConstr(
                gp.quicksum(x[(j, i, k)] for j in nodes_k if j != i) == 
                gp.quicksum(x[(i, j, k)] for j in nodes_k if j != i),
                name=f"continuity_{i}_{k}"
            )

    # (6) At most one visit per station
    for i in I:
        model.addConstr(
            gp.quicksum(x[(j, i, k)] for k in K for j in I + [f"depot_{k}"] if j != i) <= 1,
            name=f"single_visit_{i}"
        )

    # (7) Transfers bounded by visit
    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        for i in I:
            visit_var = gp.quicksum(x[(j, i, k)] for j in nodes_k if j != i)
            model.addConstr(z[(i, k)] >= -I0[i] * visit_var, name=f"transfer_lb_{i}_{k}")
            model.addConstr(z[(i, k)] <= (C[i] - I0[i]) * visit_var, name=f"transfer_ub_{i}_{k}")

    # (8) Station Dock Capacity Constraints
    for i in I:
        total_transfer = gp.quicksum(z[(i, k)] for k in K)
        model.addConstr(I0[i] + total_transfer >= 0, name=f"station_min_inv_{i}")
        model.addConstr(I0[i] + total_transfer <= C[i], name=f"station_max_inv_{i}")

    # (9) Unmet Target Shortfall Penalty
    for i in I:
        total_transfer = gp.quicksum(z[(i, k)] for k in K)
        model.addConstr(G[i] - total_transfer <= u[i], name=f"penalty_pos_{i}")
        model.addConstr(G[i] - total_transfer >= -u[i], name=f"penalty_neg_{i}")

    # (10) Vehicle Load Bound
    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        for i in I:
            visit_var = gp.quicksum(x[(j, i, k)] for j in nodes_k if j != i)
            model.addConstr(l[(i, k)] <= truck_capacity * visit_var, name=f"load_ub_{i}_{k}")

    # (11) Big-M Load Flow Balance
    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        for j in I:
            for i in nodes_k:
                if i != j:
                    model.addConstr(
                        l[(j, k)] - l[(i, k)] + z[(j, k)] >= -M * (1 - x[(i, j, k)]),
                        name=f"load_track_lb_{i}_{j}_{k}"
                    )
                    model.addConstr(
                        l[(j, k)] - l[(i, k)] + z[(j, k)] <= M * (1 - x[(i, j, k)]),
                        name=f"load_track_ub_{i}_{j}_{k}"
                    )

    # (12) Empty Return to Depot
    for k in K:
        depot_k = f"depot_{k}"
        for i in I:
            model.addConstr(
                l[(i, k)] <= truck_capacity * (1 - x[(i, depot_k, k)]),
                name=f"empty_return_{i}_{k}"
            )

    # (13) Max Travel Budget Limit
    for k in K:
        depot_k = f"depot_{k}"
        nodes_k = I + [depot_k]
        model.addConstr(
            gp.quicksum(D[(i, j, k)] * x[(i, j, k)] for i in nodes_k for j in nodes_k if i != j) <= max_distance_w,
            name=f"dist_limit_{k}"
        )

    # (14) SUBTOUR ELIMINATION (MTZ Sequence Ordering Constraints)
    for k in K:
        depot_k = f"depot_{k}"
        for i in I:
            visit_var = gp.quicksum(x[(j, i, k)] for j in I + [depot_k] if j != i)
            model.addConstr(y[(i, k)] <= len(I) * visit_var, name=f"seq_ub_{i}_{k}")
            model.addConstr(y[(i, k)] >= 1 - len(I) * (1 - x[(depot_k, i, k)]), name=f"seq_depot_{i}_{k}")
            for j in I:
                if i != j:
                    model.addConstr(
                        y[(j, k)] >= y[(i, k)] + 1 - (len(I)+1) * (1 - x[(i, j, k)]),
                        name=f"seq_step_{i}_{j}_{k}"
                    )

    from pathlib import Path
    import json,time
    out=Path('outputs_stage8');out.mkdir(exist_ok=True)
    
    for k in K:
        model.addConstr(gp.quicksum(z[i,k] for i in I)==0,name=f'explicit_conservation_{k}')
    model.update();model.write(str(out/'rebalancing.mps'))
    # Build a multi-stop warm start by inserting beneficial pickup/delivery pairs; all input stations remain eligible.
    # This is a feasible MILP start only; Gurobi can change stations, quantities and order.
    routes_seed = {k: [f'depot_{k}', f'depot_{k}'] for k in K}
    transfers_seed = {k: {} for k in K}
    used_seed = set()
    def route_distance(path, k):
        return sum(D[a,b,k] for a,b in zip(path[:-1],path[1:]) if a != b)
    def feasible_loads(path, transfers):
        load = 0
        for node in path[1:-1]:
            load -= transfers[node]
            if load < 0 or load > truck_capacity:
                return False
        return load == 0
    while True:
        best = None
        for k in K:
            path = routes_seed[k]
            old_distance = route_distance(path,k)
            load = 0
            for pos in range(1,len(path)):
                if pos > 1:
                    load -= transfers_seed[k][path[pos-1]]
                a,b = path[pos-1],path[pos]
                for i in I:
                    if i in used_seed or G[i] >= 0: continue
                    for j in I:
                        if j in used_seed or G[j] <= 0: continue
                        q = min(-G[i],G[j],truck_capacity-load)
                        if q <= 0: continue
                        extra = D[a,i,k]+D[i,j,k]+D[j,b,k]-(D[a,b,k] if a != b else 0)
                        gain = 2*beta*q-alpha*extra
                        if gain <= 1e-9 or old_distance+extra > max_distance_w+1e-9: continue
                        candidate = (-gain,extra,k,pos,i,j,q)
                        if best is None or candidate < best: best=candidate
        if best is None: break
        _,_,k,pos,i,j,q = best
        routes_seed[k][pos:pos] = [i,j]
        transfers_seed[k].update({i:-q,j:q})
        used_seed.update([i,j])
    # Improve the initial route with segment reversals at fixed transfers; recheck loads each time.
    for k in K:
        path = routes_seed[k]
        while True:
            best_path = path
            best_distance = route_distance(path,k)
            for a in range(1,len(path)-2):
                for b in range(a+1,len(path)-1):
                    candidate = path[:a]+list(reversed(path[a:b+1]))+path[b+1:]
                    dist = route_distance(candidate,k)
                    if dist < best_distance-1e-9 and feasible_loads(candidate,transfers_seed[k]):
                        best_path,best_distance=candidate,dist
            if best_path is path: break
            path=best_path
        routes_seed[k]=path
    names = [v.VarName for v in model.getVars()]
    seed = dict.fromkeys(names,0.0)
    for i in I: seed[u[i].VarName]=abs(G[i])
    for k in K:
        path=routes_seed[k]
        if len(path)==2: continue
        load=0
        for a,b in zip(path[:-1],path[1:]): seed[x[a,b,k].VarName]=1
        for order,i in enumerate(path[1:-1],1):
            q=transfers_seed[k][i];load-=q
            seed[z[i,k].VarName]=q;seed[l[i,k].VarName]=load
            seed[y[i,k].VarName]=order;seed[u[i].VarName]=abs(G[i]-q)
    # Check the warm start against every MILP row, including MTZ and Big-M constraints.
    maximum_violation=0.0
    for constraint in model.getConstrs():
        row=model.getRow(constraint)
        lhs=sum(row.getCoeff(t)*seed[row.getVar(t).VarName] for t in range(row.size()))
        violation=(abs(lhs-constraint.RHS) if constraint.Sense=='=' else
                   max(lhs-constraint.RHS,0) if constraint.Sense=='<' else max(constraint.RHS-lhs,0))
        maximum_violation=max(maximum_violation,violation)
    for v in model.getVars():
        value=seed[v.VarName]
        assert v.LB-1e-6 <= value <= v.UB+1e-6
        if v.VType != GRB.CONTINUOUS: assert abs(value-round(value)) <= 1e-6
    assert maximum_violation <= 1e-6, maximum_violation
    seed_objective=sum(v.Obj*seed[v.VarName] for v in model.getVars())
    seed_audit={'objective':seed_objective,'maximum_constraint_violation':maximum_violation,
                'constraints_checked':model.NumConstrs,'routes':routes_seed,
                'transfers':transfers_seed,'visited_stations':len(used_seed)}
    (out/'warm_start_audit.json').write_text(json.dumps(seed_audit,ensure_ascii=False,indent=2))
    print('Multi-stop warm start:',len(used_seed),'stations; objective',seed_objective,
          '; maximum full-model constraint violation',maximum_violation,flush=True)
    for v in model.getVars(): v.Start=seed[v.VarName]

    values={};solver='Gurobi';gap=None;bound=None
    start=time.perf_counter()
    model.Params.LogFile=str(out/'gurobi.log')
    model.optimize()
    status={GRB.OPTIMAL:'Optimal',GRB.TIME_LIMIT:'Time limit reached',GRB.INFEASIBLE:'Infeasible'}.get(model.Status,str(model.Status))
    runtime=model.Runtime
    if model.SolCount == 0:
        raise RuntimeError('Gurobi returned no feasible incumbent. Check the log; do not reuse old results.')
    values={v.VarName:v.X for v in model.getVars()}
    gap=model.MIPGap;bound=model.ObjBound
    result={'solver':solver,'status':status,'runtime_seconds':runtime,'has_feasible_solution':bool(values),
            'gap':gap,'objective_bound':bound,'stations':len(I),'trucks':len(K),'routes':{},'station_results':[]}
    if values:
        val=lambda var:values[var.VarName]
        distance=0.0
        for k in K:
            depot=f'depot_{k}';nodes=I+[depot]
            arcs=[(i,j) for i in nodes for j in nodes if i!=j and val(x[i,j,k])>0.5]
            nxt=dict(arcs);path=[depot];cur=depot
            for step in range(len(I)+2):
                if cur not in nxt:break
                cur=nxt[cur];path.append(cur)
                if cur==depot:break
            assert not arcs or (path[-1]==depot and len(path)-1==len(arcs))
            dist=sum(D[i,j,k] for i,j in arcs);distance+=dist
            result['routes'][k]={'path':path,'distance_km':dist,
                'transfers':{i:round(val(z[i,k])) for i in I if i in path},
                'loads':{i:val(l[i,k]) for i in I if i in path}}
        for i in I:
            net=round(sum(val(z[i,k]) for k in K));final=I0[i]+net
            assert 0<=final<=C[i]
            result['station_results'].append({'id':i,'I0':I0[i],'net_delivery':net,'final_inventory':final,'target':I0[i]+G[i],'mismatch':abs(G[i]-net)})
        assert sum(r['net_delivery'] for r in result['station_results'])==0
        mismatch=sum(r['mismatch'] for r in result['station_results'])
        result.update(total_distance_km=distance,total_mismatch=mismatch,objective=alpha*distance+beta*mismatch,
            baseline_mismatch=sum(abs(v) for v in G.values()),baseline_objective=beta*sum(abs(v) for v in G.values()))
    result['warm_start_objective']=seed_objective
    result['improved_over_warm_start']=bool(values) and result['objective'] < seed_objective-1e-6
    (out/'solution.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
    return result


In [ ]:
import json
from pathlib import Path
data8=json.loads(Path('outputs_stage6/model_station_inputs.json').read_text())
depots8=json.loads(Path('outputs_stage7/selected_depots.json').read_text())
solution8=solve_citi_bike_rebalancing(data8,depots8,truck_capacity=25,max_distance_w=25,alpha=1.5,beta=5)
print(json.dumps({k:v for k,v in solution8.items() if k not in ['routes','station_results']},ensure_ascii=False,indent=2))


### Step 8 results: Run Gurobi locally
Targets have changed, so previous routes and gaps are not current results. Use the actual solver output and newly generated `solution.json`.


## Step 9: Validate the new-target Gurobi result
Read current Step 6 inputs and Step 8 results. Check loads, inventories, visits, distances and conservation. Targets now come from fixed-fleet event replay and must not be explained by the superseded half-capacity rule. This section does not retrain prediction models.


In [ ]:
import csv, json, math
from pathlib import Path
import matplotlib.pyplot as plt

out9 = Path('outputs_stage9')
out9.mkdir(exist_ok=True)
sol9 = json.loads(Path('outputs_stage8/solution.json').read_text())
dep9 = json.loads(Path('outputs_stage7/selected_depots.json').read_text())
st9 = json.loads(Path('outputs_stage6/model_station_inputs.json').read_text())['stations']
stations9 = {s['id']: s for s in st9}
nodes9 = {**stations9, **{d['depot_id']: d for d in dep9}}
results9 = {s['id']: s for s in sol9['station_results']}
Q9, W9, alpha9, beta9 = 25, 25, 1.5, 5

def distance9(a, b):
    return round(84 * abs(a['lon']-b['lon']) + 111 * abs(a['lat']-b['lat']), 3)

def csv9(name, rows):
    with (out9 / name).open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

# Independently reconstruct loads, distance and inventories from exported routes.
assert sol9['has_feasible_solution']
assert len(stations9) == len(st9) == len(results9) == len(sol9['station_results'])
assert set(stations9) == set(results9)
assert set(sol9['routes']) == {d['truck_id'] for d in dep9}
seen9, route_rows9 = set(), []
delivery9 = dict.fromkeys(stations9, 0)
total_distance9, moved9 = 0, 0
for d in dep9:
    truck = d['truck_id']
    r = sol9['routes'][truck]
    path = r['path']
    assert path[0] == path[-1] == d['depot_id']
    assert set(r['transfers']) == set(path[1:-1])
    load, distance = 0, 0
    route_rows9.append({'truck':truck, 'order':0, 'station_id':path[0], 'name':d['name'],
                        'net_delivery':0, 'load_after':0, 'distance_so_far_km':0})
    for order, (a,b) in enumerate(zip(path[:-1], path[1:]), 1):
        distance += distance9(nodes9[a], nodes9[b])
        z = 0
        if b in stations9:
            assert b not in seen9
            seen9.add(b)
            z = r['transfers'][b]
            assert abs(z-round(z)) < 1e-6
            delivery9[b] += z
            load -= z  # Positive transfer delivers to a station and reduces truck load.
            assert math.isclose(load, r['loads'][b], abs_tol=1e-6)
            moved9 += max(-z, 0)
        else:
            assert b == d['depot_id'] and order == len(path)-1
        assert 0 <= load <= Q9
        route_rows9.append({'truck':truck, 'order':order, 'station_id':b, 'name':nodes9[b]['name'],
                           'net_delivery':z, 'load_after':load, 'distance_so_far_km':round(distance,3)})
    assert load == 0 and sum(r['transfers'].values()) == 0
    assert distance <= W9 + 1e-6
    assert math.isclose(distance, r['distance_km'], abs_tol=1e-6)
    total_distance9 += distance

station_rows9 = []
for sid, s in stations9.items():
    r = results9[sid]
    final = s['I0'] + delivery9[sid]
    assert 0 <= final <= s['C']
    assert r['I0'] == s['I0'] and r['target'] == s['I_star']
    assert r['net_delivery'] == delivery9[sid] and r['final_inventory'] == final
    assert r['mismatch'] == abs(final-s['I_star'])
    station_rows9.append({'station_id':sid, 'name':s['name'], 'effective_capacity':s['C'],
        'midnight_inventory':s['I0'], 'target_inventory':s['I_star'], 'net_delivery_total':delivery9[sid],
        'final_inventory':final, 'absolute_mismatch_before':abs(s['I0']-s['I_star']),
        'absolute_mismatch_after':abs(final-s['I_star']), 'visited':sid in seen9})

initial9 = sum(s['I0'] for s in st9)
target9 = sum(s['I_star'] for s in st9)
before9 = sum(s['absolute_mismatch_before'] for s in station_rows9)
after9 = sum(s['absolute_mismatch_after'] for s in station_rows9)
lower9 = abs(target9-initial9)
cost9 = alpha9*total_distance9 + beta9*after9
assert sum(delivery9.values()) == 0
assert sum(s['final_inventory'] for s in station_rows9) == initial9
assert after9 >= lower9
assert before9 == sol9['baseline_mismatch'] and after9 == sol9['total_mismatch']
assert math.isclose(total_distance9, sol9['total_distance_km'], abs_tol=1e-6)
assert math.isclose(cost9, sol9['objective'], abs_tol=1e-6)
assert math.isclose(beta9*before9, sol9['baseline_objective'], abs_tol=1e-6)
assert cost9 <= beta9*before9 + 1e-6
summary9 = {'station_count':len(st9), 'visited_station_count':len(seen9), 'bikes_moved':moved9,
    'initial_inventory_total':initial9, 'target_inventory_total':target9, 'conservation_mismatch_lower_bound':lower9,
    'mismatch_before':before9, 'mismatch_after':after9, 'mismatch_reduction':before9-after9,
    'approximate_total_distance_km':round(total_distance9,3), 'transport_cost':round(alpha9*total_distance9,4),
    'mismatch_penalty_cost':beta9*after9, 'objective':round(cost9,4),
    'no_rebalancing_objective':beta9*before9, 'objective_reduction':round(beta9*before9-cost9,4),
    'proven_optimal':False, 'checks_passed':True}
csv9('route_steps.csv', route_rows9)
csv9('station_before_after.csv', station_rows9)
csv9('summary.csv', [summary9])
checks9 = [{'check':x, 'result':'passed'} for x in [
    'Inputs and results cover the same complete station set', 'Each truck starts and returns at its own depot',
    'Each station is visited at most once across the fleet', 'Integer transfers agree with reconstructed route loads',
    'Loads remain within 0-25 and trucks start/finish empty', 'Each truck stays within the 25 km approximate distance budget',
    'Final station inventory is nonnegative and within effective capacity', 'Regional bike total is conserved',
    'Targets and initial stock agree with inputs', 'Recomputed distances, mismatch and objective agree',
    'Mismatch respects the conservation lower bound', 'Objective is no worse than no rebalancing']]
csv9('validation_checks.csv', checks9)
(out9/'stage9_summary.json').write_text(json.dumps(summary9,ensure_ascii=False,indent=2))
print(json.dumps(summary9, ensure_ascii=False, indent=2))
print('\nAll 12 check categories passed. Positive transfer means delivery; negative means pickup.')
for row in route_rows9:
    print(row['truck'], row['order'], row['name'], 'net_delivery_total', row['net_delivery'], 'load', row['load_after'])




### 9.2 Both routes on a street basemap
Read saved Steps 6–8 results without repeating prediction or optimization. Install `contextily pyproj adjustText` if needed. Show all input stations, both depots, actual visit order, names and transfers; titles use actual results.

Projection follows the [contextily documentation](https://contextily.readthedocs.io/en/latest/reference.html). Connecting lines remain visiting sequences, not street-network navigation.


In [ ]:
# Read the actual saved Step 8 result; do not predict or solve again.
from pathlib import Path
import json, csv, textwrap, math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyproj import Transformer
import contextily as ctx
from adjustText import adjust_text

MAP_ROOT = Path.cwd()  # Use the project directory containing outputs_stage6, outputs_stage7 and outputs_stage8.
required = ['outputs_stage6/model_station_inputs.json',
            'outputs_stage7/selected_depots.json','outputs_stage8/solution.json']
for relative in required:
    if not (MAP_ROOT/relative).is_file():
        raise FileNotFoundError(f'Current directory {MAP_ROOT} is missing {relative}. Open this notebook inside the fully extracted project folder.')
stations_map = json.loads((MAP_ROOT/required[0]).read_text())['stations']
depots_map = json.loads((MAP_ROOT/required[1]).read_text())
solution_map = json.loads((MAP_ROOT/required[2]).read_text())
if not solution_map.get('has_feasible_solution'):
    raise ValueError('No feasible Step 8 result has been saved; solve the model first.')
by_id_map = {s['id']:s for s in stations_map}
nodes_map = {**by_id_map, **{d['depot_id']:d for d in depots_map}}
result_ids_map = {s['id'] for s in solution_map['station_results']}
assert all(r['target']==by_id_map[r['id']]['I_star'] and r['I0']==by_id_map[r['id']]['I0'] for r in solution_map['station_results']), 'This result uses old targets; rerun Step 8.'
assert result_ids_map == set(by_id_map), 'Input and result station sets differ; check Steps 6-8 files.'
projection_map = Transformer.from_crs(4326,3857,always_xy=True)
xy_map = {sid:projection_map.transform(s['lon'],s['lat']) for sid,s in nodes_map.items()}
out_map = MAP_ROOT/'outputs_stage9';out_map.mkdir(exist_ok=True)
colors_map = ['#16a36a','#943fc3','#e08a22','#2078b4']
LABEL_ALL_STATIONS = False  # True labels every station; 166 names may overlap.
fig_map,ax_map = plt.subplots(figsize=(17,19))
points_map = list(xy_map.values())
xs_map,ys_map = zip(*points_map)
padding_map = 240
ax_map.set_xlim(min(xs_map)-padding_map,max(xs_map)+padding_map)
ax_map.set_ylim(min(ys_map)-padding_map,max(ys_map)+padding_map)
print('Loading street basemap; an internet connection is required.')
try:
    ctx.add_basemap(ax_map,source=ctx.providers.Esri.WorldStreetMap,
                    zoom=14,alpha=.68,attribution_size=7,zorder=0)
except Exception as exc:
    plt.close(fig_map)
    raise RuntimeError('Street basemap failed. Check the connection and rerun this cell; no model rerun is needed.') from exc

legend_map=[]
for condition,color,label in [
    (lambda s:s['G']<0,'#235778','Surplus station (G < 0, pickup)'),
    (lambda s:s['G']>0,'#de4027','Deficit station (G > 0, delivery)'),
    (lambda s:s['G']==0,'#989b9f','Balanced station (G = 0)')]:
    subset=[s for s in stations_map if condition(s)]
    ax_map.scatter([xy_map[s['id']][0] for s in subset],
                   [xy_map[s['id']][1] for s in subset],
                   c=color,s=38,edgecolors='white',linewidths=.8,zorder=4)
    legend_map.append(Line2D([],[],marker='o',linestyle='',color=color,label=label,markersize=7))
labels_map=[];steps_map=[];visited_map=set()
for truck_index,d in enumerate(depots_map):
    truck=d['truck_id'];route=solution_map['routes'][truck];path=route['path']
    color=colors_map[truck_index%len(colors_map)]
    assert not path or path[0]==path[-1]==d['depot_id']
    if len(path)>1:
        coords=[xy_map[sid] for sid in path]
        ax_map.plot([v[0] for v in coords],[v[1] for v in coords],color=color,lw=2.5,alpha=.9,zorder=2)
        for a,b in zip(coords[:-1],coords[1:]):
            start=(a[0]+.40*(b[0]-a[0]),a[1]+.40*(b[1]-a[1]))
            end=(a[0]+.64*(b[0]-a[0]),a[1]+.64*(b[1]-a[1]))
            ax_map.annotate('',xy=end,xytext=start,arrowprops={'arrowstyle':'-|>','color':color,'lw':2,'mutation_scale':13},zorder=3)
    legend_map.append(Line2D([],[],color=color,lw=3,label=f"{truck}: {len(path[1:-1])} stops, {route['distance_km']:.3f} km"))
    dx,dy=xy_map[d['depot_id']]
    ax_map.scatter(dx,dy,marker='*',s=650,c='#ffd82b',edgecolors='black',linewidths=1.7,zorder=7)
    labels_map.append(ax_map.text(dx+65,dy+85,f"Depot {truck_index+1}\n"+d['name'].replace('Depot '+str(truck_index+1)+' (candidate near ','').rstrip(')'),
                                 fontsize=9,weight='bold',ha='center',bbox={'facecolor':'#ffe458','edgecolor':'black','boxstyle':'round,pad=.3'},zorder=8))
    for order,sid in enumerate(path[1:-1],1):
        visited_map.add(sid);s=by_id_map[sid];xx,yy=xy_map[sid]
        transfer=route['transfers'].get(sid,0)
        label=f"{truck_index+1}.{order}  "+textwrap.fill(s['name'],24)+f"\nG={s['G']:+g} | transfer {transfer:+g} bikes"
        labels_map.append(ax_map.text(xx+75,yy+(55 if order%2 else -100),label,fontsize=7.5,ha='center',va='center',
                            bbox={'facecolor':'white','edgecolor':color,'alpha':.95,'boxstyle':'round,pad=.3'},zorder=6))
        steps_map.append({'truck':truck,'order':order,'station_id':sid,'name':s['name'],
                          'G':s['G'],'net_delivery':transfer,'load_after':route['loads'][sid]})
if LABEL_ALL_STATIONS:
    for sid,s in by_id_map.items():
        if sid not in visited_map:
            xx,yy=xy_map[sid]
            labels_map.append(ax_map.text(xx,yy,textwrap.fill(s['name'],24)+f"\nG={s['G']:+g}",fontsize=6,
                bbox={'facecolor':'white','edgecolor':'#aaa','alpha':.8},zorder=5))
ax_map.legend(handles=legend_map,loc='upper left',fontsize=10,framealpha=.97)
solver_map=solution_map.get('solver','Unknown')
gap_map=solution_map.get('gap')
gap_text_map=f'{gap_map:.2%}' if gap_map is not None else 'N/A'
fig_map.suptitle('Citi Bike Overnight Rebalancing — Midtown Manhattan\n'
    f"{len(stations_map)} input stations | {len(visited_map)} visited | solver: {solver_map} | gap: {gap_text_map}\n"
    f"Case: {stations_map[0]['case_date']} midnight NYC | target deviation: {solution_map['total_mismatch']} bikes | distance: {solution_map['total_distance_km']:.3f} km",
    fontsize=16,weight='bold',y=.98)
ax_map.set_title('Transfers: + delivery / - pickup. Lines show visit order; road routing is not computed.',fontsize=10,pad=12)
ax_map.set_aspect('equal');ax_map.set_xlabel('Longitude');ax_map.set_ylabel('Latitude')
reverse_map=Transformer.from_crs(3857,4326,always_xy=True)
ax_map.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,pos:f'{reverse_map.transform(x,0)[0]:.3f}'))
ax_map.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,pos:f'{reverse_map.transform(0,y)[1]:.3f}'))
fig_map.tight_layout(rect=[0,0,.99,.93])
adjust_text(labels_map,ax=ax_map,x=list(xs_map),y=list(ys_map),
            arrowprops={'arrowstyle':'-','color':'#555','lw':.6},
            expand=(1.08,1.15),time_lim=5)
fig_map.savefig(out_map/'routes_street_map.png',dpi=220,bbox_inches='tight')
fig_map.savefig(out_map/'routes_street_map.pdf',bbox_inches='tight')
if steps_map:
    with (out_map/'map_route_details.csv').open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(steps_map[0]));writer.writeheader();writer.writerows(steps_map)
print(f'Recorded solver: {solver_map}; visited {len(visited_map)} stations. Map saved to outputs_stage9.')
if solver_map != 'Gurobi':
    print('The solver recorded in solution.json is not Gurobi; this map does not relabel old results as Gurobi output.')
plt.show()


The map uses the current `solution.json`. A route may visit only part of the input set; visit counts cannot be promised in advance.


### Stop-by-stop operations
After running the validation cell, inspect the printed sequence. Detailed transfers and loads are exported to `outputs_stage9/route_steps.csv`.


### 9.3 Inventory changes at visited stations
Horizontal bars show before/after inventory; black marks show targets. The CSV retains all 166 stations. Unvisited stations are still part of the model.


In [ ]:
visited9=[s for s in station_rows9 if s['visited']]
fig,ax=plt.subplots(figsize=(10,max(5,.38*len(visited9))))
y=list(range(len(visited9)))
ax.barh([i-.2 for i in y],[s['midnight_inventory'] for s in visited9],height=.4,label='Before')
ax.barh([i+.2 for i in y],[s['final_inventory'] for s in visited9],height=.4,label='After')
ax.scatter([s['target_inventory'] for s in visited9],y,marker='|',s=180,c='black',label='Target',zorder=4)
ax.set_yticks(y,[s['station_id']+' '+s['name'] for s in visited9]);ax.invert_yaxis()
ax.set(xlabel='Bikes',title='All visited stations: inventory before / after / target')
ax.legend();fig.tight_layout();fig.savefig(out9/'inventory.png',dpi=150);plt.show()


The chart shows stations visited in the current solution. Target mismatch is not unserved trip demand. Cumulative bikes moved may exceed 25, but simultaneous truck load must not exceed 25.


### 9.4 Regional absolute target mismatch
Absolute mismatch is $\sum_i |I_i^{final}-I_i^*|$, measured in bikes. It is neither unmet orders nor forecast error.


In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(['No rebalancing','Gurobi candidate'],[before9,after9],color=['#929ca7','#1768ac'])
for i,v in enumerate([before9,after9]):ax.text(i,v,str(v),ha='center',va='bottom')
ax.set(ylabel='Absolute inventory deviation (bikes)',title='Deviation from the new fleet-conserving targets')
fig.tight_layout();fig.savefig(out9/'mismatch.png',dpi=160);plt.show()


The target total equals the initial total, so aggregate conservation no longer forces a positive mismatch. Remaining deviations depend on route feasibility, operating limits, cost trade-offs and solver quality.


### Reporting and further validation
Target allocation and routing are separate stages. The first allocates the existing fleet; Gurobi then chooses how to move bikes. Replay metrics are evaluated on the historical scenarios used to construct targets, not an independent test. Comparing old/new replay scores under different fleet totals is not a fair performance comparison. No minimum visit count was imposed and the demand forecasts were unchanged.

Validate service outcomes and target sensitivity. Report the effective-capacity proxy, completed-trip demand proxy, equal blocked pickup/return loss weights, ignored overnight rides and unverified road/time feasibility.
